
# Stage 05 — External Validation and H3 (Canonical Final)

**Canonical role.** This notebook finalizes the Stage-05 scientific layer from the already-successful, immutable Stage-05 inference cache. It deliberately performs **no model inference, no calibration fitting, no target tuning, and no cue selection**.

It does four things only:

1. cryptographically verifies and reuses the frozen source/BanglaClick/BaitBuster prediction artifacts and H3-P perturbation-pair inference artifact;
2. recomputes H1/H2 under a strict, fail-closed estimability policy;
3. executes the validated H3 estimability repair as the sole authoritative H3 implementation;
4. finalizes H3-P from the immutable pair-level `delta_z` cache using all 5,000 locked cluster-bootstrap draws.

The original development Stage-05 statistical outputs are treated only as **parity references**. They are never copied into the canonical result package.

**Scientific boundary.** Non-estimability is a scientific result, not a pipeline failure. No bootstrap draw is silently discarded, no rejection resampling is used, and no H3 dimension/family is removed post hoc.



## Input contract

A previously successful Stage-05 development output must be mounted read-only. The cache must contain:

- `LOCKED_SOURCE_TEST_PREDICTIONS.parquet`
- `BANGLACLICK_PLATFORM_PREDICTIONS.parquet`
- `BAITBUSTER_HUMAN_PREDICTIONS.parquet`
- `H3_PERTURBATION_PAIRS.parquet`
- `H3_PERTURBATION_EXCLUSIONS.csv`
- the original Stage-05 manifest/checksum/status files
- original `STATISTICAL_SUMMARY.json` and `H3_PERTURBATION_RESULTS.json` **only for parity checks**

The canonical notebook has no inference fallback. Missing or ambiguous cache identity causes a hard stop rather than a GPU rerun.


In [1]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter, defaultdict
import hashlib, importlib.metadata as im, json, math, os, platform, re, shutil, subprocess, sys, time

import numpy as np
import pandas as pd
from scipy.special import logit
from scipy.stats import chi2
import statsmodels.api as sm
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
from statsmodels.stats.multitest import multipletests

STAGE_ID = "05"
STAGE_NAME = "05_external_validation_and_H3_vFINAL"
CANONICAL_VERSION = "SWARABYANJAN-STAGE05-CANONICAL-vFINAL-1.0.1"
PROTOCOL_VERSION = "SWARABYANJAN-CROSSPLATFORM-v1.0.0"
H3_REPAIR_VERSION = "SWARABYANJAN-H3-ESTIMABILITY-REPAIR-v1.0.0"
H1H2_REPAIR_VERSION = "SWARABYANJAN-H1H2-ESTIMABILITY-REPAIR-v1.0.1"
H3P_FINALIZATION_VERSION = "SWARABYANJAN-H3P-FINALIZATION-v1.0.0"
# Compatibility alias used by the validated H3 repair cell; do not change repair semantics.
REPAIR_VERSION = H3_REPAIR_VERSION

BOOTSTRAP_REPS = 5000
BOOTSTRAP_SEED = 20260906
FDR_Q = 0.05
SEEDS = [42, 137, 2718, 31415, 65537]
MODEL_KEYS = ["ebm", "tfidf_lr", "banglabert", "xlmr"]
CUE_FAMILIES = [
    "curiosity_information_withholding",
    "direct_address_imperative",
    "exaggeration_superlative",
    "interrogative_structure",
    "punctuation_intensity",
    "sensational_emotional_loading",
    "specificity_numerals",
    "vague_deictic_reference",
]
PLATFORMS = ["news", "facebook", "youtube", "twitter"]
PERT_FAMS = ["curiosity_information_withholding", "direct_address_imperative", "punctuation_intensity"]

WORK_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
SEARCH_ROOTS = [
    Path("/kaggle/input"),
    Path("/kaggle/working"),
    Path.cwd(),
]
BRANCH_RULE = "RULE_A"
BASE_STAGE05_MANIFEST_SHA = os.environ.get("SWARABYANJAN_BASE_STAGE05_MANIFEST_SHA256")
OUT_ROOT = WORK_ROOT / "03_results" / STAGE_NAME
if OUT_ROOT.exists() and any(OUT_ROOT.iterdir()):
    raise RuntimeError(f"Refusing to overwrite existing canonical Stage-05 output: {OUT_ROOT}")
OUT_ROOT.mkdir(parents=True, exist_ok=True)


def utc_now():
    return datetime.now(timezone.utc).isoformat()


def _jsonable(obj):
    """Normalize NumPy/Pandas scalar containers without silently serializing NaN/Inf."""
    if obj is None or isinstance(obj, (str, bool, int)):
        return obj
    if isinstance(obj, float):
        if not math.isfinite(obj):
            raise ValueError(f"Non-finite float cannot be serialized: {obj}")
        return obj
    if isinstance(obj, np.bool_):
        return bool(obj)
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        x = float(obj)
        if not math.isfinite(x):
            raise ValueError(f"Non-finite NumPy float cannot be serialized: {obj}")
        return x
    if isinstance(obj, np.ndarray):
        return [_jsonable(x) for x in obj.tolist()]
    if isinstance(obj, (list, tuple)):
        return [_jsonable(x) for x in obj]
    if isinstance(obj, dict):
        return {str(k): _jsonable(v) for k, v in obj.items()}
    if isinstance(obj, Path):
        return str(obj)
    if obj is pd.NA:
        return None
    raise TypeError(f"Unsupported JSON type: {type(obj).__name__}")


def cjb(obj):
    return json.dumps(_jsonable(obj), ensure_ascii=False, sort_keys=True, separators=(",", ":"), allow_nan=False).encode("utf-8")


def write_json(path, obj):
    Path(path).write_text(
        json.dumps(_jsonable(obj), ensure_ascii=False, sort_keys=True, indent=2, allow_nan=False) + "\n",
        encoding="utf-8",
        newline="\n",
    )


def write_text(path, text):
    Path(path).write_text(str(text), encoding="utf-8", newline="\n")


def sha256_file(path, chunk=8 * 1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def set_status(status, message, details=None):
    write_json(
        OUT_ROOT / "STAGE_STATUS.json",
        {
            "stage_id": STAGE_ID,
            "canonical_version": CANONICAL_VERSION,
            "status": status,
            "message": message,
            "timestamp_utc": utc_now(),
            "details": details or {},
        },
    )
    print(f"STAGE 05 STATUS: {status} — {message}", flush=True)


def hard_stop(message, details=None):
    set_status("HARD_STOP", message, details)
    raise RuntimeError(message)


def pkg(name):
    try:
        return im.version(name)
    except Exception:
        return None


def git_state():
    try:
        head = subprocess.check_output(["git", "rev-parse", "HEAD"], stderr=subprocess.DEVNULL, text=True).strip()
        status = subprocess.check_output(["git", "status", "--porcelain"], stderr=subprocess.DEVNULL, text=True)
        return {"available": True, "head": head, "dirty": bool(status.strip())}
    except Exception:
        return {"available": False, "head": None, "dirty": None}

ENVIRONMENT_PREFLIGHT = {
    "timestamp_utc": utc_now(),
    "mode": "CPU_ONLY_VERIFIED_IMMUTABLE_INFERENCE_CACHE",
    "python": sys.version,
    "platform": platform.platform(),
    "cpu_count": os.cpu_count(),
    "packages": {n: pkg(n) for n in ["numpy", "pandas", "pyarrow", "scipy", "statsmodels", "scikit-learn"]},
    "model_inference_permitted": False,
    "calibration_refit_permitted": False,
}
write_json(OUT_ROOT / "ENVIRONMENT_PREFLIGHT.json", ENVIRONMENT_PREFLIGHT)
set_status("RUNNING", "Canonical Stage-05 initialization complete")
print("Canonical Stage-05 output root:", OUT_ROOT, flush=True)

STAGE 05 STATUS: RUNNING — Canonical Stage-05 initialization complete


Canonical Stage-05 output root: /mnt/data/swarabyanjan_branch_exec/RULE_A/03_results/05_external_validation_and_H3_vFINAL



## Immutable inference-cache discovery and verification

The development output is used only as an immutable inference cache. Its original H1/H2/H3/H3-P inferential conclusions are not trusted by inheritance. Cache identity is established from the frozen prediction and perturbation artifacts; multiple identical mounts are collapsed, while multiple distinct identities hard-stop.


In [2]:
REQUIRED_CACHE_FILES = [
    "LOCKED_SOURCE_TEST_PREDICTIONS.parquet",
    "BANGLACLICK_PLATFORM_PREDICTIONS.parquet",
    "BAITBUSTER_HUMAN_PREDICTIONS.parquet",
    "H3_PERTURBATION_PAIRS.parquet",
    "H3_PERTURBATION_EXCLUSIONS.csv",
    "STATISTICAL_SUMMARY.json",
    "H3_PERTURBATION_RESULTS.json",
    "STAGE05_ARTIFACT_MANIFEST.json",
    "STAGE05_SHA256SUMS.txt",
    "STAGE_STATUS.json",
]
OPTIONAL_CACHE_EVIDENCE = [
    "FROZEN_INPUT_DISCOVERY.json",
    "BANGLACLICK_SCHEMA_PROBE.json",
    "BANGLACLICK_SCHEMA_REPORT.json",
    "BANGLACLICK_DIAGNOSTIC_CORROBORATION.json",
    "EXTERNAL_DATA_ADAPTER_REPORT.json",
]


def parse_sumfile(path):
    out = {}
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        h, rel = line.split(None, 1)
        rel = rel.lstrip("*").strip().replace("\\", "/")
        if not re.fullmatch(r"[0-9a-fA-F]{64}", h):
            raise ValueError(f"Malformed checksum line: {line}")
        if rel in out:
            raise ValueError(f"Duplicate checksum path: {rel}")
        out[rel] = h.lower()
    return out


def discover_inference_cache_root():
    candidates = set()
    for root in SEARCH_ROOTS:
        if not root.exists():
            continue
        for p in root.rglob("STAGE05_ARTIFACT_MANIFEST.json"):
            d = p.parent.resolve()
            if d == OUT_ROOT.resolve():
                continue
            if not all((d / name).is_file() for name in REQUIRED_CACHE_FILES):
                continue
            try:
                manifest = json.loads((d / "STAGE05_ARTIFACT_MANIFEST.json").read_text(encoding="utf-8"))
            except Exception:
                continue
            # Do not recursively use a previously canonicalized output as the development inference source.
            if manifest.get("canonical_stage05") is True:
                continue
            candidates.add(d)

    if not candidates:
        hard_stop(
            "Successful development Stage-05 inference cache not found",
            {"required_files": REQUIRED_CACHE_FILES, "searched_roots": [str(x) for x in SEARCH_ROOTS]},
        )

    identities = defaultdict(list)
    for d in sorted(candidates, key=str):
        identity = {name: sha256_file(d / name) for name in [
            "LOCKED_SOURCE_TEST_PREDICTIONS.parquet",
            "BANGLACLICK_PLATFORM_PREDICTIONS.parquet",
            "BAITBUSTER_HUMAN_PREDICTIONS.parquet",
            "H3_PERTURBATION_PAIRS.parquet",
            "H3_PERTURBATION_EXCLUSIONS.csv",
        ]}
        key = hashlib.sha256(cjb(identity)).hexdigest()
        identities[key].append((d, identity))

    if len(identities) != 1:
        hard_stop(
            "Multiple distinct Stage-05 inference-cache identities found",
            {"candidate_roots": [str(x) for x in sorted(candidates, key=str)], "identity_count": len(identities)},
        )
    group = next(iter(identities.values()))
    chosen, identity = sorted(group, key=lambda x: str(x[0]))[0]
    if len(group) > 1:
        print(f"Identical inference-cache mounts collapsed: {len(group)}; using {chosen}", flush=True)
    return chosen, identity


_forced_branch_cache = os.environ.get("SWARABYANJAN_BRANCH_CACHE_ROOT")
if _forced_branch_cache:
    d = Path(_forced_branch_cache).resolve()
    if not d.is_dir() or not all((d/n).is_file() for n in REQUIRED_CACHE_FILES):
        hard_stop("Branch Stage-05 cache root is incomplete", {"root": str(d)})
    bm = json.loads((d / "STAGE05_ARTIFACT_MANIFEST.json").read_text(encoding="utf-8"))
    if bm.get("canonical_stage05") is not False or bm.get("branch_rule") != BRANCH_RULE:
        hard_stop("Branch Stage-05 cache identity mismatch", {"root": str(d), "manifest_canonical_stage05": bm.get("canonical_stage05"), "manifest_branch_rule": repr(bm.get("branch_rule")), "notebook_branch_rule": repr(BRANCH_RULE)})
    if os.environ.get("SWARABYAN_BASE_STAGE05_MANIFEST_SHA256") and bm.get("base_canonical_stage05_manifest_sha256") != os.environ.get("SWARABYAN_BASE_STAGE05_MANIFEST_SHA256"):
        hard_stop("Branch Stage-05 lineage does not point to canonical base manifest")
    identity = {name: sha256_file(d / name) for name in [
        "LOCKED_SOURCE_TEST_PREDICTIONS.parquet", "BANGLACLICK_PLATFORM_PREDICTIONS.parquet",
        "BAITBUSTER_HUMAN_PREDICTIONS.parquet", "H3_PERTURBATION_PAIRS.parquet",
        "H3_PERTURBATION_EXCLUSIONS.csv",
    ]}
    SOURCE_ROOT, CACHE_IDENTITY = d, identity
else:
    SOURCE_ROOT, CACHE_IDENTITY = discover_inference_cache_root()
SOURCE_STATUS = json.loads((SOURCE_ROOT / "STAGE_STATUS.json").read_text(encoding="utf-8"))
if SOURCE_STATUS.get("status") != "PASS":
    hard_stop("Development Stage-05 cache source is not PASS", {"status": SOURCE_STATUS.get("status")})

SOURCE_MANIFEST = json.loads((SOURCE_ROOT / "STAGE05_ARTIFACT_MANIFEST.json").read_text(encoding="utf-8"))
SOURCE_MANIFEST_SHA256 = sha256_file(SOURCE_ROOT / "STAGE05_ARTIFACT_MANIFEST.json")
SOURCE_SUMS = parse_sumfile(SOURCE_ROOT / "STAGE05_SHA256SUMS.txt")
# Compatibility aliases consumed by the validated H3 repair cell.
source_manifest_sha = SOURCE_MANIFEST_SHA256
SOURCE_IDENTITY = {
    "manifest": SOURCE_MANIFEST_SHA256,
    "source_predictions": sha256_file(SOURCE_ROOT / "LOCKED_SOURCE_TEST_PREDICTIONS.parquet"),
    "banglaclick_predictions": sha256_file(SOURCE_ROOT / "BANGLACLICK_PLATFORM_PREDICTIONS.parquet"),
    "baitbuster_predictions": sha256_file(SOURCE_ROOT / "BAITBUSTER_HUMAN_PREDICTIONS.parquet"),
}

recorded_payload = SOURCE_MANIFEST.get("manifest_payload_sha256")
if recorded_payload:
    tmp = dict(SOURCE_MANIFEST)
    tmp.pop("manifest_payload_sha256", None)
    observed_payload = hashlib.sha256(cjb(tmp)).hexdigest()
    if observed_payload != recorded_payload:
        hard_stop("Inference-cache Stage-05 manifest payload seal mismatch")

# Every cache artifact must agree with both available provenance surfaces.
for name in REQUIRED_CACHE_FILES:
    p = SOURCE_ROOT / name
    observed = sha256_file(p)
    manifest_hash = SOURCE_MANIFEST.get("artifacts", {}).get(name)
    checksum_hash = SOURCE_SUMS.get(name)
    if manifest_hash is not None and observed != manifest_hash:
        hard_stop("Inference-cache artifact disagrees with source manifest", {"artifact": name})
    if checksum_hash is not None and observed != checksum_hash:
        hard_stop("Inference-cache artifact disagrees with source checksum file", {"artifact": name})

# Copy only immutable scientific inference products needed by the canonical stage.
IMMUTABLE_REUSE_FILES = [
    "LOCKED_SOURCE_TEST_PREDICTIONS.parquet",
    "BANGLACLICK_PLATFORM_PREDICTIONS.parquet",
    "BAITBUSTER_HUMAN_PREDICTIONS.parquet",
    "H3_PERTURBATION_PAIRS.parquet",
    "H3_PERTURBATION_EXCLUSIONS.csv",
]
for name in IMMUTABLE_REUSE_FILES:
    shutil.copyfile(SOURCE_ROOT / name, OUT_ROOT / name)
    if sha256_file(OUT_ROOT / name) != sha256_file(SOURCE_ROOT / name):
        hard_stop("Immutable cache copy hash mismatch", {"artifact": name})

CACHE_PROVENANCE = {
    "schema_version": 1,
    "canonical_version": CANONICAL_VERSION,
    "source_root": str(SOURCE_ROOT),
    "source_manifest_sha256": SOURCE_MANIFEST_SHA256,
    "source_status_sha256": sha256_file(SOURCE_ROOT / "STAGE_STATUS.json"),
    "immutable_cache_identity": CACHE_IDENTITY,
    "upstream_stage03_checksum_manifest_sha256": SOURCE_MANIFEST.get("upstream_stage03_checksum_manifest_sha256"),
    "upstream_stage04_checksum_manifest_sha256": SOURCE_MANIFEST.get("upstream_stage04_checksum_manifest_sha256"),
    "frozen_content_identity_sha256": SOURCE_MANIFEST.get("frozen_content_identity_sha256"),
    "canonical_zip_sha256_provenance": SOURCE_MANIFEST.get("canonical_zip_sha256_provenance"),
    "model_inference_rerun": False,
    "calibration_refit": False,
    "development_statistics_inherited": False,
}
write_json(OUT_ROOT / "INFERENCE_CACHE_PROVENANCE.json", CACHE_PROVENANCE)
print("Verified immutable inference cache:", SOURCE_ROOT, flush=True)


Verified immutable inference cache: /mnt/data/swarabyanjan_branch_exec/RULE_A/input_cache


## Prediction and perturbation-cache schema gates

In [3]:
SRC_P = pd.read_parquet(OUT_ROOT / "LOCKED_SOURCE_TEST_PREDICTIONS.parquet")
BC_P = pd.read_parquet(OUT_ROOT / "BANGLACLICK_PLATFORM_PREDICTIONS.parquet")
BB_P = pd.read_parquet(OUT_ROOT / "BAITBUSTER_HUMAN_PREDICTIONS.parquet")
PERT = pd.read_parquet(OUT_ROOT / "H3_PERTURBATION_PAIRS.parquet")
EXCL = pd.read_csv(OUT_ROOT / "H3_PERTURBATION_EXCLUSIONS.csv")

required_pred = {
    "row_id", "title", "platform", "label", "cluster_id", "dataset", "model", "seed",
    "logit", "p_uncalibrated", "temperature", "p_calibrated", "normalized_title_sha256",
    "token_count", "latin_code_mixing_proportion", *CUE_FAMILIES,
}
expected_news_rows = {"RULE_A":456, "RULE_B":426}.get(os.environ.get("SWARABYANJAN_BRANCH_RULE"), 1500)
expected_item_rows = {"source": 2784, "banglaclick": 1961 + 1000 + 500 + expected_news_rows, "baitbuster": 10000}
for name, df in [("source", SRC_P), ("banglaclick", BC_P), ("baitbuster", BB_P)]:
    missing = sorted(required_pred - set(df.columns))
    if missing:
        hard_stop("Prediction cache schema incomplete", {"domain": name, "missing": missing})
    if df[["row_id", "cluster_id"]].isna().any().any():
        hard_stop("Prediction cache has missing identity", {"domain": name})
    if set(df["model"].astype(str).unique()) != set(MODEL_KEYS):
        hard_stop("Prediction cache model panel mismatch", {"domain": name})
    if set(df["seed"].astype(int).unique()) != set(SEEDS):
        hard_stop("Prediction cache seed panel mismatch", {"domain": name})
    if not set(df["label"].astype(int).unique()).issubset({0, 1}):
        hard_stop("Prediction cache labels are not binary", {"domain": name})
    for col in ["p_uncalibrated", "p_calibrated", "logit", "temperature"]:
        if not np.isfinite(df[col].to_numpy(float)).all():
            hard_stop("Prediction cache has non-finite numeric values", {"domain": name, "column": col})
    if ((df["p_calibrated"].to_numpy(float) <= 0) | (df["p_calibrated"].to_numpy(float) >= 1)).any():
        hard_stop("Calibrated probabilities outside open interval (0,1)", {"domain": name})
    for (model, seed), g in df.groupby(["model", "seed"], sort=True):
        if len(g) != expected_item_rows[name] or g["row_id"].nunique() != expected_item_rows[name]:
            hard_stop("Prediction model/seed row contract mismatch", {"domain": name, "model": str(model), "seed": int(seed)})

static_cols = ["title", "platform", "label", "cluster_id", "normalized_title_sha256", "token_count", "latin_code_mixing_proportion", *CUE_FAMILIES]
def validate_static_replication(df, domain):
    canonical = (
        df[(df["model"] == "ebm") & (df["seed"].astype(int) == SEEDS[0])]
        .sort_values("row_id", kind="stable").set_index("row_id")[static_cols]
    )
    for (model, seed), g in df.groupby(["model", "seed"], sort=True):
        x = g.sort_values("row_id", kind="stable").set_index("row_id")[static_cols]
        if not x.index.equals(canonical.index) or not x.equals(canonical):
            hard_stop("Static prediction fields differ across model/seed replicas", {"domain": domain, "model": str(model), "seed": int(seed)})
    return canonical.reset_index()

src_static = validate_static_replication(SRC_P, "source")
bc_static = validate_static_replication(BC_P, "banglaclick")
bb_static = validate_static_replication(BB_P, "baitbuster")
if set(bc_static["platform"].astype(str).unique()) != set(PLATFORMS):
    hard_stop("BanglaClick platform set mismatch", {"observed": sorted(map(str, bc_static["platform"].unique()))})
if not src_static["cluster_id"].astype(str).str.startswith("source_row:").all():
    hard_stop("Source TEST clustering differs from locked row fallback")
if not bc_static["cluster_id"].astype(str).str.startswith("bc:").all():
    hard_stop("BanglaClick clustering identity differs from locked Stage-05 form")
if not bb_static["cluster_id"].astype(str).str.startswith("bb:").all():
    hard_stop("BaitBuster clustering identity differs from locked channel form")

required_pert = {"domain", "row_id", "cluster_id", "cue", "original_title", "neutralized_title", "model", "seed", "delta_z"}
missing = sorted(required_pert - set(PERT.columns))
if missing:
    hard_stop("H3-P inference cache schema incomplete", {"missing": missing})
if PERT.empty or PERT.duplicated(["domain", "row_id", "cue", "model", "seed"]).any():
    hard_stop("H3-P inference cache empty or duplicate pair identities detected")
if set(PERT["model"].astype(str).unique()) != {"ebm", "banglabert"}:
    hard_stop("H3-P model panel mismatch")
if set(PERT["seed"].astype(int).unique()) != set(SEEDS):
    hard_stop("H3-P seed panel mismatch")
if set(PERT["cue"].astype(str).unique()) != set(PERT_FAMS):
    hard_stop("H3-P cue-family panel mismatch")
if not np.isfinite(PERT["delta_z"].to_numpy(float)).all():
    hard_stop("H3-P inference cache contains non-finite delta_z")

# All model/seed replicas must contain the identical frozen pair universe.
base_keys = None
for (model, seed), g in PERT.groupby(["model", "seed"], sort=True):
    keys = list(zip(g["domain"].astype(str), g["row_id"].astype(str), g["cue"].astype(str), g["cluster_id"].astype(str)))
    keys = sorted(keys)
    if base_keys is None:
        base_keys = keys
    elif keys != base_keys:
        hard_stop("H3-P pair universe differs across model/seed inference replicas", {"model": str(model), "seed": int(seed)})

write_json(OUT_ROOT / "CACHE_SCHEMA_VALIDATION.json", {
    "status": "PASS",
    "source_items_per_model_seed": 2784,
    "banglaclick_items_per_model_seed": expected_item_rows["banglaclick"],
    "baitbuster_items_per_model_seed": 10000,
    "h3p_unique_pairs_per_model_seed": len(base_keys),
    "models": MODEL_KEYS,
    "seeds": SEEDS,
    "cue_families": CUE_FAMILIES,
    "h3p_families": PERT_FAMS,
})
print("Immutable cache schema validation: PASS", flush=True)

Immutable cache schema validation: PASS



## H1/H2 deterministic metrics and sensitivity analyses

Point metrics are recomputed from frozen predictions. The existing source-prior and structural-composition sensitivity definitions are retained unchanged.


In [4]:
# ---------- Metric definitions and H1/H2 ----------
print("[H1/H2 metrics] START — deterministic point metrics and sensitivity metrics", flush=True)
def balanced_brier(y,p,w=None):
    y=np.asarray(y,int);p=np.asarray(p,float);w=None if w is None else np.asarray(w,float);vals=[]
    for c in [0,1]:
        m=y==c
        if not m.any():return None
        vals.append(float(np.average((p[m]-y[m])**2,weights=None if w is None else w[m])))
    return 0.5*sum(vals)
def calib_is(y,p,w=None):
    p=np.clip(np.asarray(p,float),1e-7,1-1e-7);X=sm.add_constant(logit(p),has_constant="add")
    try:
        fit=sm.GLM(np.asarray(y,int),X,family=sm.families.Binomial(),freq_weights=w).fit()
        return float(fit.params[0]),float(fit.params[1]),None
    except (ValueError, np.linalg.LinAlgError, RuntimeError) as exc:
        return None,None,f"{type(exc).__name__}: {exc}"
def metric_bundle(y,p,w=None):
    y=np.asarray(y,int);p=np.clip(np.asarray(p,float),1e-7,1-1e-7);w=None if w is None else np.asarray(w,float)
    if len(np.unique(y))<2:
        return {"estimable":False,"reason":"single_class","auroc":None,"auprc":None,"nll":None,"brier":None,"balanced_brier":None,"calibration_intercept":None,"calibration_slope":None,"calibration_fit_issue":"single_class","f1_at_0_50":None}
    if w is not None and (not np.isfinite(w).all() or w.sum()<=0 or any(w[y==c].sum()<=0 for c in [0,1])):
        return {"estimable":False,"reason":"nonpositive_or_class-degenerate_weights","auroc":None,"auprc":None,"nll":None,"brier":None,"balanced_brier":None,"calibration_intercept":None,"calibration_slope":None,"calibration_fit_issue":"nonpositive_or_class-degenerate_weights","f1_at_0_50":None}
    ci,cs,fit_issue=calib_is(y,p,w)
    nll=float(np.average(-(y*np.log(p)+(1-y)*np.log(1-p)),weights=w));brier=float(np.average((p-y)**2,weights=w))
    return {"estimable":True,"auroc":float(roc_auc_score(y,p,sample_weight=w)),"auprc":float(average_precision_score(y,p,sample_weight=w)),
            "nll":nll,"brier":brier,"balanced_brier":balanced_brier(y,p,w),"calibration_intercept":ci,"calibration_slope":cs,
            "calibration_fit_issue":fit_issue,"f1_at_0_50":float(f1_score(y,p>=0.5,sample_weight=w))}
def _relative_change_failclosed(target_value, source_value, source_name):
    """Return locked relative change or an explicit invalid reason; never emit Inf/NaN."""
    source_value = float(source_value)
    target_value = float(target_value)
    if not np.isfinite(source_value):
        return None, f"source_{source_name}_nonfinite"
    if source_value <= 0:
        return None, f"source_{source_name}_nonpositive"
    if not np.isfinite(target_value):
        return None, f"target_{source_name}_nonfinite"
    value = (target_value - source_value) / source_value
    if not np.isfinite(value):
        return None, f"{source_name}_relative_change_nonfinite"
    return float(value), None


def source_prior_weights(y,target_pi):
    y=np.asarray(y,int); pi=float(y.mean());
    if pi in {0,1}:return np.ones(len(y))
    return np.where(y==1,target_pi/pi,(1-target_pi)/(1-pi))
def structural_weights(source_features,target_features):
    # Unsupervised composition standardization: source quartiles of token count and source median split of code-mixing.
    q=np.unique(np.quantile(source_features["token_count"],[0,.25,.5,.75,1]))
    if len(q)<2:q=np.array([-np.inf,np.inf])
    else:q[0],q[-1]=-np.inf,np.inf
    smed=float(source_features["latin_code_mixing_proportion"].median())
    def strata(df):
        tb=np.digitize(df["token_count"].to_numpy(),q[1:-1],right=True); cb=(df["latin_code_mixing_proportion"].to_numpy()>smed).astype(int);return tb*2+cb
    ss=strata(source_features);ts=strata(target_features);sp=Counter(ss);tp=Counter(ts);N=len(ss);M=len(ts)
    return np.array([(sp.get(x,0)/N)/(tp.get(x,1)/M) if tp.get(x,0)>0 and sp.get(x,0)>0 else 0.0 for x in ts],float)

# Domain frames: each seed/model appears repeatedly but unique item rows are stable.
domain_frames={"source_test":SRC_P}
for plat in ["news","facebook","youtube","twitter"]:domain_frames[f"banglaclick:{plat}"]=BC_P[BC_P["platform"]==plat].copy()
domain_frames["baitbuster:youtube_human"]=BB_P
source_item=SRC_P[(SRC_P.model=="ebm")&(SRC_P.seed==SEEDS[0])].drop_duplicates("row_id")
source_pi=float(source_item.label.mean())

metric_rows=[]
for dname,df in domain_frames.items():
    for (m,s),g in df.groupby(["model","seed"],sort=True):
        g=g.sort_values("row_id",kind="stable");y=g.label.to_numpy();pu=g.p_uncalibrated.to_numpy();pc=g.p_calibrated.to_numpy()
        rec={"domain":dname,"model":m,"seed":int(s),"n":len(g),"prevalence":float(y.mean()),
             "uncalibrated":metric_bundle(y,pu),"calibrated":metric_bundle(y,pc)}
        if dname!="source_test":
            wp=source_prior_weights(y,source_pi);rec["source_prior_standardized"]={"uncalibrated":metric_bundle(y,pu,wp),"calibrated":metric_bundle(y,pc,wp)}
            tgt=g.drop_duplicates("row_id")[["token_count","latin_code_mixing_proportion"]]
            srcf=source_item[["token_count","latin_code_mixing_proportion"]];ws=structural_weights(srcf,tgt)
            # expand item weights in this single model/seed group one-to-one
            rec["structural_composition_standardized"]={"calibrated":metric_bundle(y,pc,ws)}
        metric_rows.append(rec)

# H1 support screen per model/seed/target using source calibrated comparison.
source_lookup={(r["model"],r["seed"]):r for r in metric_rows if r["domain"]=="source_test"}
h1_screen=[];h2_screen=[]
for r in metric_rows:
    if r["domain"]=="source_test":continue
    srcm=source_lookup[(r["model"],r["seed"])]
    auc_drop=srcm["calibrated"]["auroc"]-r["calibrated"]["auroc"]
    nll_rel,nll_rel_reason=_relative_change_failclosed(r["calibrated"]["nll"],srcm["calibrated"]["nll"],"nll")
    bbs_rel,bbs_rel_reason=_relative_change_failclosed(r["calibrated"]["balanced_brier"],srcm["calibrated"]["balanced_brier"],"balanced_brier")
    if nll_rel_reason or bbs_rel_reason:
        hard_stop("H1 point-estimate relative change is undefined", {"domain":r["domain"],"model":r["model"],"seed":int(r["seed"]),"nll_reason":nll_rel_reason,"balanced_brier_reason":bbs_rel_reason})
    h1_screen.append({"domain":r["domain"],"model":r["model"],"seed":r["seed"],"target_auroc":r["calibrated"]["auroc"],"auroc_drop":auc_drop,
                      "nll_relative_change":nll_rel,"balanced_brier_relative_change":bbs_rel,
                      "descriptive_h1_condition":bool(r["calibrated"]["auroc"]>=.70 and auc_drop<=.05 and (nll_rel>=.10 or bbs_rel>=.10))})
    gs=srcm["uncalibrated"]["nll"]-srcm["calibrated"]["nll"];gt=r["uncalibrated"]["nll"]-r["calibrated"]["nll"]
    h2_screen.append({"domain":r["domain"],"model":r["model"],"seed":r["seed"],"source_nll_gain":gs,"target_nll_gain":gt,"target_minus_source_gain":gt-gs})
print(f"[H1/H2 metrics] COMPLETE — metric_rows={len(metric_rows)}", flush=True)


[H1/H2 metrics] START — deterministic point metrics and sensitivity metrics


[H1/H2 metrics] COMPLETE — metric_rows=120



## Strict H1/H2 cluster bootstrap

The bootstrap law remains the locked ordinary cluster bootstrap. The repair changes only inferential handling:

- **H1:** AUROC and balanced Brier require both classes. If any of the 5,000 locked source/target draws makes the H1 vector undefined, the setting is `NON_ESTIMABLE_UNDER_LOCKED_BOOTSTRAP`; no CI is computed from the surviving subset.
- **H2:** the NLL-gain statistic is mathematically defined for single-class draws. Therefore H2 is evaluated on **all 5,000 ordinary draws**, rather than being accidentally discarded when the H1 vector is undefined. This is not rejection resampling; it removes the old computational coupling between H1's two-class requirement and H2's proper-loss statistic.

This distinction is essential for BaitBuster: strict H1 may be non-estimable while H2 remains fully estimable.

- **Relative-change denominator hardening:** source NLL and source balanced Brier must be finite and strictly positive on every locked replicate. Exact endpoint-specific invalid reasons are recorded; no infinite or undefined ratio is emitted. Overall H1 remains conservative even though endpoint-specific estimability is now exposed.


In [5]:
BOOTSTRAP_PROGRESS_EVERY = 1000
BOOTSTRAP_PHASE_LIMIT_SECONDS = 7200

class ClusterBootstrapPlan:
    def __init__(self, cluster_ids, name):
        clusters = np.asarray(cluster_ids).astype(str)
        if len(clusters) == 0:
            hard_stop("Cannot build cluster bootstrap plan for empty frame", {"name": name})
        uniq = np.array(sorted(set(clusters)))
        code = {u: i for i, u in enumerate(uniq)}
        self.row_cluster = np.asarray([code[u] for u in clusters], dtype=np.int32)
        self.cluster_labels = uniq
        self.n_clusters = int(len(uniq))
        self.n_rows = int(len(clusters))
        self.name = str(name)

    def draw_row_counts(self, rng):
        sampled_codes = rng.choice(self.n_clusters, size=self.n_clusters, replace=True)
        cluster_multiplicity = np.bincount(sampled_codes, minlength=self.n_clusters)
        return cluster_multiplicity[self.row_cluster]


def _phase_guard(phase, phase_start, completed=None, total=None, context=None):
    elapsed = time.monotonic() - phase_start
    if elapsed > BOOTSTRAP_PHASE_LIMIT_SECONDS:
        hard_stop(
            f"{phase} exceeded runtime guard",
            {"elapsed_seconds": elapsed, "completed": completed, "total": total, "context": context or {}},
        )
    return elapsed


def _progress(phase, done, total, phase_start, detail=""):
    elapsed = _phase_guard(phase, phase_start, done, total, {"detail": detail})
    rate = done / elapsed if elapsed > 0 else None
    eta = (total - done) / rate if rate and rate > 0 else None
    print(f"[{phase}] {done}/{total} elapsed={elapsed:.1f}s eta={eta:.1f}s {detail}".rstrip(), flush=True)


def complete_ci(values):
    a = np.asarray(values, float)
    if len(a) != BOOTSTRAP_REPS or not np.isfinite(a).all():
        raise ValueError("Confirmatory CI requires all 5,000 finite locked bootstrap replicates")
    return [float(np.percentile(a, 2.5)), float(np.percentile(a, 97.5))]


def complete_p(values):
    a = np.asarray(values, float)
    if len(a) != BOOTSTRAP_REPS or not np.isfinite(a).all():
        raise ValueError("Confirmatory p-value requires all 5,000 finite locked bootstrap replicates")
    return float(min(1.0, 2 * min(np.mean(a <= 0), np.mean(a >= 0))))


def _auc_setup(y, p):
    y = np.asarray(y, int); p = np.asarray(p, float)
    order = np.argsort(p, kind="mergesort")
    ps = p[order]
    starts = np.r_[0, np.flatnonzero(ps[1:] != ps[:-1]) + 1]
    return order, starts


def _auc_from_row_counts(y, row_counts, setup):
    y = np.asarray(y, int); w = np.asarray(row_counts, float)
    order, starts = setup
    ys = y[order]; ws = w[order]
    pos = np.add.reduceat(ws * ys, starts)
    neg = np.add.reduceat(ws * (1 - ys), starts)
    P = float(pos.sum()); N = float(neg.sum())
    if P <= 0 or N <= 0:
        return np.nan
    neg_before = np.cumsum(neg) - neg
    return float(np.sum(pos * (neg_before + 0.5 * neg)) / (P * N))


def _weighted_mean(v, w):
    v = np.asarray(v, float); w = np.asarray(w, float)
    den = float(w.sum())
    return float(np.dot(v, w) / den) if den > 0 else np.nan


def _two_class_reasons(y, w, prefix):
    y = np.asarray(y, int); w = np.asarray(w, float)
    reasons = []
    if float(w[y == 0].sum()) <= 0:
        reasons.append(f"{prefix}_missing_class_0")
    if float(w[y == 1].sum()) <= 0:
        reasons.append(f"{prefix}_missing_class_1")
    return reasons


def _h1_metrics(y, p, row_counts, auc_setup, loss, sqerr):
    reasons = _two_class_reasons(y, row_counts, "sample")
    if reasons:
        return None
    y = np.asarray(y, int); w = np.asarray(row_counts, float)
    auc = _auc_from_row_counts(y, w, auc_setup)
    nll = _weighted_mean(loss, w)
    w0 = w[y == 0]; w1 = w[y == 1]
    bb = 0.5 * (_weighted_mean(sqerr[y == 0], w0) + _weighted_mean(sqerr[y == 1], w1))
    if not np.isfinite([auc, nll, bb]).all():
        return None
    return auc, nll, bb

# One immutable cluster plan per domain.
domain_boot_plans = {}
domain_boot_row_ids = {}
for dname, df in domain_frames.items():
    base = df[(df.model == MODEL_KEYS[0]) & (df.seed == SEEDS[0])].sort_values("row_id", kind="stable").reset_index(drop=True)
    domain_boot_plans[dname] = ClusterBootstrapPlan(base.cluster_id.to_numpy(), dname)
    domain_boot_row_ids[dname] = base.row_id.astype(str).tolist()

rng = np.random.default_rng(BOOTSTRAP_SEED)
stat_rows = []
targets = [d for d in domain_frames if d != "source_test"]
total_settings = len(targets) * len(MODEL_KEYS) * len(SEEDS)
phase_start = time.monotonic()
setting_done = 0

for target_name in targets:
    target_df = domain_frames[target_name]
    for model in MODEL_KEYS:
        for seed in SEEDS:
            setting_done += 1
            detail = f"setting={setting_done}/{total_settings} target={target_name} model={model} seed={seed}"
            s = SRC_P[(SRC_P.model == model) & (SRC_P.seed == seed)].sort_values("row_id", kind="stable").reset_index(drop=True)
            t = target_df[(target_df.model == model) & (target_df.seed == seed)].sort_values("row_id", kind="stable").reset_index(drop=True)
            if s.row_id.astype(str).tolist() != domain_boot_row_ids["source_test"]:
                hard_stop("Source row order differs from bootstrap plan", {"model": model, "seed": seed})
            if t.row_id.astype(str).tolist() != domain_boot_row_ids[target_name]:
                hard_stop("Target row order differs from bootstrap plan", {"target": target_name, "model": model, "seed": seed})

            sy = s.label.to_numpy(int); ty = t.label.to_numpy(int)
            sp = np.clip(s.p_calibrated.to_numpy(float), 1e-7, 1 - 1e-7)
            tp = np.clip(t.p_calibrated.to_numpy(float), 1e-7, 1 - 1e-7)
            su = np.clip(s.p_uncalibrated.to_numpy(float), 1e-7, 1 - 1e-7)
            tu = np.clip(t.p_uncalibrated.to_numpy(float), 1e-7, 1 - 1e-7)
            s_loss = -(sy * np.log(sp) + (1 - sy) * np.log(1 - sp))
            t_loss = -(ty * np.log(tp) + (1 - ty) * np.log(1 - tp))
            su_loss = -(sy * np.log(su) + (1 - sy) * np.log(1 - su))
            tu_loss = -(ty * np.log(tu) + (1 - ty) * np.log(1 - tu))
            s_sq = (sp - sy) ** 2; t_sq = (tp - ty) ** 2
            s_auc_setup = _auc_setup(sy, sp); t_auc_setup = _auc_setup(ty, tp)
            spl = domain_boot_plans["source_test"]; tpl = domain_boot_plans[target_name]

            h1_aucd = []; h1_nlld = []; h1_bbd = []
            h1_auroc_invalid_reasons = defaultdict(int)
            h1_nll_invalid_reasons = defaultdict(int)
            h1_balanced_brier_invalid_reasons = defaultdict(int)
            h1_overall_invalid_reasons = defaultdict(int)
            h1_overall_valid_replicates = 0
            h2_source_gain = []; h2_target_gain = []; h2_gain_diff = []
            h2_invalid_reasons = defaultdict(int)

            for rep in range(1, BOOTSTRAP_REPS + 1):
                sw = spl.draw_row_counts(rng); tw = tpl.draw_row_counts(rng)

                # H1 AUROC endpoint: both classes are required in source and target.
                auc_reasons = _two_class_reasons(sy, sw, "source") + _two_class_reasons(ty, tw, "target")
                auc_value = None
                if auc_reasons:
                    for reason in sorted(set(auc_reasons)):
                        h1_auroc_invalid_reasons[reason] += 1
                else:
                    sauc = _auc_from_row_counts(sy, sw, s_auc_setup)
                    tauc = _auc_from_row_counts(ty, tw, t_auc_setup)
                    if not np.isfinite(sauc):
                        h1_auroc_invalid_reasons["source_auroc_nonfinite"] += 1
                    elif not np.isfinite(tauc):
                        h1_auroc_invalid_reasons["target_auroc_nonfinite"] += 1
                    else:
                        auc_value = float(sauc - tauc)
                        if not np.isfinite(auc_value):
                            h1_auroc_invalid_reasons["auroc_drop_nonfinite"] += 1
                            auc_value = None
                        else:
                            h1_aucd.append(auc_value)

                # H1 relative NLL endpoint: denominator must be finite and strictly positive.
                snll = _weighted_mean(s_loss, sw)
                tnll = _weighted_mean(t_loss, tw)
                nll_value, nll_reason = _relative_change_failclosed(tnll, snll, "nll")
                if nll_reason is not None:
                    h1_nll_invalid_reasons[nll_reason] += 1
                else:
                    h1_nlld.append(nll_value)

                # H1 balanced-Brier relative-change endpoint: both classes plus a valid positive source denominator.
                bb_reasons = _two_class_reasons(sy, sw, "source") + _two_class_reasons(ty, tw, "target")
                bb_value = None
                if bb_reasons:
                    for reason in sorted(set(bb_reasons)):
                        h1_balanced_brier_invalid_reasons[reason] += 1
                else:
                    sw0 = sw[sy == 0]; sw1 = sw[sy == 1]
                    tw0 = tw[ty == 0]; tw1 = tw[ty == 1]
                    sbb = 0.5 * (_weighted_mean(s_sq[sy == 0], sw0) + _weighted_mean(s_sq[sy == 1], sw1))
                    tbb = 0.5 * (_weighted_mean(t_sq[ty == 0], tw0) + _weighted_mean(t_sq[ty == 1], tw1))
                    bb_value, bb_reason = _relative_change_failclosed(tbb, sbb, "balanced_brier")
                    if bb_reason is not None:
                        h1_balanced_brier_invalid_reasons[bb_reason] += 1
                        bb_value = None
                    else:
                        h1_bbd.append(bb_value)

                # Conservative overall H1 remains valid only when all three endpoints are valid on this locked draw.
                rep_overall_reasons = []
                if auc_value is None:
                    rep_overall_reasons.append("auroc_invalid")
                if nll_value is None:
                    rep_overall_reasons.append(nll_reason or "nll_relative_change_invalid")
                if bb_value is None:
                    rep_overall_reasons.append("balanced_brier_relative_change_invalid")
                if rep_overall_reasons:
                    h1_overall_invalid_reasons["+".join(sorted(set(rep_overall_reasons)))] += 1
                else:
                    h1_overall_valid_replicates += 1

                # H2: NLL gain is defined even for a single-class cluster resample.
                snll_all = _weighted_mean(s_loss, sw)
                tnll_all = _weighted_mean(t_loss, tw)
                gs = _weighted_mean(su_loss, sw) - snll_all
                gt = _weighted_mean(tu_loss, tw) - tnll_all
                vals = np.asarray([gs, gt, gt - gs], float)
                if not np.isfinite(vals).all():
                    h2_invalid_reasons["nonfinite_nll_gain"] += 1
                else:
                    h2_source_gain.append(float(gs)); h2_target_gain.append(float(gt)); h2_gain_diff.append(float(gt - gs))

                if rep % 100 == 0:
                    _phase_guard("H1/H2 bootstrap", phase_start, rep, BOOTSTRAP_REPS, {"setting": detail})
                if rep % BOOTSTRAP_PROGRESS_EVERY == 0:
                    _progress("H1/H2 bootstrap", rep, BOOTSTRAP_REPS, phase_start, detail)

            h1_auroc_estimable = len(h1_aucd) == BOOTSTRAP_REPS and not h1_auroc_invalid_reasons
            h1_nll_estimable = len(h1_nlld) == BOOTSTRAP_REPS and not h1_nll_invalid_reasons
            h1_bb_estimable = len(h1_bbd) == BOOTSTRAP_REPS and not h1_balanced_brier_invalid_reasons
            h1_estimable = h1_auroc_estimable and h1_nll_estimable and h1_bb_estimable and h1_overall_valid_replicates == BOOTSTRAP_REPS
            h1_invalid = BOOTSTRAP_REPS - int(h1_overall_valid_replicates)
            h2_invalid = int(sum(h2_invalid_reasons.values()))
            h2_estimable = h2_invalid == 0 and len(h2_gain_diff) == BOOTSTRAP_REPS

            combined_h1_reasons = defaultdict(int)
            for endpoint, reason_map in [
                ("auroc", h1_auroc_invalid_reasons),
                ("nll_relative_change", h1_nll_invalid_reasons),
                ("balanced_brier_relative_change", h1_balanced_brier_invalid_reasons),
            ]:
                for reason, count in reason_map.items():
                    combined_h1_reasons[f"{endpoint}:{reason}"] += int(count)

            row = {
                "domain": target_name, "model": model, "seed": int(seed),
                "bootstrap_attempts": BOOTSTRAP_REPS,
                "h1_valid_replicates": int(h1_overall_valid_replicates),
                "h1_invalid_replicates": int(h1_invalid),
                "h1_invalid_reason_counts": dict(sorted(combined_h1_reasons.items())),
                "h1_overall_invalid_combination_counts": dict(sorted(h1_overall_invalid_reasons.items())),
                "h1_inferential_status": "ESTIMABLE" if h1_estimable else "NON_ESTIMABLE_UNDER_LOCKED_BOOTSTRAP",
                "auroc_estimable": bool(h1_auroc_estimable),
                "nll_relative_change_estimable": bool(h1_nll_estimable),
                "balanced_brier_relative_change_estimable": bool(h1_bb_estimable),
                "auroc_valid_replicates": int(len(h1_aucd)),
                "auroc_invalid_replicates": int(BOOTSTRAP_REPS - len(h1_aucd)),
                "auroc_invalid_reason_counts": dict(sorted(h1_auroc_invalid_reasons.items())),
                "nll_relative_change_valid_replicates": int(len(h1_nlld)),
                "nll_relative_change_invalid_replicates": int(BOOTSTRAP_REPS - len(h1_nlld)),
                "nll_relative_change_invalid_reason_counts": dict(sorted(h1_nll_invalid_reasons.items())),
                "balanced_brier_relative_change_valid_replicates": int(len(h1_bbd)),
                "balanced_brier_relative_change_invalid_replicates": int(BOOTSTRAP_REPS - len(h1_bbd)),
                "balanced_brier_relative_change_invalid_reason_counts": dict(sorted(h1_balanced_brier_invalid_reasons.items())),
                "auroc_drop_ci95": complete_ci(h1_aucd) if h1_auroc_estimable else None,
                "nll_relative_change_ci95": complete_ci(h1_nlld) if h1_nll_estimable else None,
                "balanced_brier_relative_change_ci95": complete_ci(h1_bbd) if h1_bb_estimable else None,
                "h2_valid_replicates": int(len(h2_gain_diff)),
                "h2_invalid_replicates": h2_invalid,
                "h2_invalid_reason_counts": dict(sorted(h2_invalid_reasons.items())),
                "h2_inferential_status": "ESTIMABLE" if h2_estimable else "NON_ESTIMABLE_UNDER_LOCKED_BOOTSTRAP",
                "h2_source_gain_ci95": complete_ci(h2_source_gain) if h2_estimable else None,
                "h2_target_gain_ci95": complete_ci(h2_target_gain) if h2_estimable else None,
                "h2_target_minus_source_gain_ci95": complete_ci(h2_gain_diff) if h2_estimable else None,
                "h2_target_minus_source_gain_p": complete_p(h2_gain_diff) if h2_estimable else None,
            }
            stat_rows.append(row)
            print(
                f"[H1/H2] COMPLETE {detail} h1={row['h1_inferential_status']} "
                f"({row['h1_valid_replicates']}/5000) h2={row['h2_inferential_status']} "
                f"({row['h2_valid_replicates']}/5000)", flush=True,
            )

print(f"[H1/H2 bootstrap] COMPLETE — settings={len(stat_rows)} elapsed={time.monotonic()-phase_start:.1f}s", flush=True)

[H1/H2 bootstrap] 1000/5000 elapsed=0.2s eta=1.0s setting=1/100 target=banglaclick:news model=ebm seed=42


[H1/H2 bootstrap] 2000/5000 elapsed=0.5s eta=0.7s setting=1/100 target=banglaclick:news model=ebm seed=42


[H1/H2 bootstrap] 3000/5000 elapsed=0.7s eta=0.5s setting=1/100 target=banglaclick:news model=ebm seed=42


[H1/H2 bootstrap] 4000/5000 elapsed=1.0s eta=0.2s setting=1/100 target=banglaclick:news model=ebm seed=42


[H1/H2 bootstrap] 5000/5000 elapsed=1.2s eta=0.0s setting=1/100 target=banglaclick:news model=ebm seed=42


...

[H1/H2 bootstrap] 3000/5000 elapsed=182.4s eta=121.6s setting=100/100 target=baitbuster:youtube_human model=xlmr seed=65537


[H1/H2 bootstrap] 4000/5000 elapsed=183.2s eta=45.8s setting=100/100 target=baitbuster:youtube_human model=xlmr seed=65537


[H1/H2 bootstrap] 5000/5000 elapsed=184.0s eta=0.0s setting=100/100 target=baitbuster:youtube_human model=xlmr seed=65537


[H1/H2] COMPLETE setting=100/100 target=baitbuster:youtube_human model=xlmr seed=65537 h1=NON_ESTIMABLE_UNDER_LOCKED_BOOTSTRAP (4950/5000) h2=ESTIMABLE (5000/5000)


[H1/H2 bootstrap] COMPLETE — settings=100 elapsed=184.0s



## H1/H2 parity gate and authoritative result finalization

For the 80 BanglaClick settings, the development implementation had all 5,000 two-class draws and therefore should be numerically identical to the repaired implementation. This is used as a hard parity gate. BaitBuster H1 is intentionally not required to match the old conditional-survivor intervals; those intervals are superseded. H2 is recomputed on all ordinary draws and is not conditioned on H1 estimability.


In [6]:
OLD_SUMMARY = json.loads((SOURCE_ROOT / "STATISTICAL_SUMMARY.json").read_text(encoding="utf-8"))
old_rows = OLD_SUMMARY.get("h1_h2_bootstrap", OLD_SUMMARY.get("h1_h2_rows", []))
old_lookup = {(r["domain"], r["model"], int(r["seed"])): r for r in old_rows}
new_lookup = {(r["domain"], r["model"], int(r["seed"])): r for r in stat_rows}


def _same_num(a, b, atol=1e-12):
    if a is None or b is None:
        return a is None and b is None
    return bool(np.isclose(float(a), float(b), rtol=0.0, atol=atol))


def _same_pair(a, b, atol=1e-12):
    return isinstance(a, list) and isinstance(b, list) and len(a) == len(b) == 2 and all(_same_num(x, y, atol) for x, y in zip(a, b))

parity_rows = []
for key, nr in new_lookup.items():
    if key not in old_lookup:
        hard_stop("Original H1/H2 parity reference missing setting", {"setting": key})
    old = old_lookup[key]
    old_valid = int(old.get("h1_valid_replicates", old.get("bootstrap_valid_two_class", -1)))
    is_branch_run = bool(BRANCH_RULE)
    count_match = old_valid == nr["h1_valid_replicates"]
    if not count_match and not is_branch_run:
        hard_stop("H1 bootstrap valid-draw count differs from development execution", {"setting": key, "old": old_valid, "new": nr["h1_valid_replicates"]})
    if key[0].startswith("banglaclick:") and not is_branch_run:
        fields = [
            "auroc_drop_ci95", "nll_relative_change_ci95", "balanced_brier_relative_change_ci95",
            "h2_source_gain_ci95", "h2_target_gain_ci95", "h2_target_minus_source_gain_ci95",
        ]
        ci_match = all(_same_pair(old.get(f), nr.get(f)) for f in fields)
        p_match = _same_num(old.get("h2_target_minus_source_gain_p"), nr.get("h2_target_minus_source_gain_p"))
        if not (ci_match and p_match and old_valid == BOOTSTRAP_REPS):
            hard_stop("BanglaClick H1/H2 parity failed", {"setting": key, "ci_match": ci_match, "p_match": p_match, "old_valid": old_valid})
        parity_status = "EXACT_NUMERICAL_PARITY"
    elif is_branch_run:
        parity_status = "BRANCH_EXPECTED_DIFFERENCE"
    else:
        parity_status = "BAITBUSTER_REPAIR_EXPECTED_DIFFERENCE" if old_valid < BOOTSTRAP_REPS else "FULL_DRAW_PARITY_ELIGIBLE"
    parity_rows.append({"domain": key[0], "model": key[1], "seed": key[2], "old_two_class_valid": old_valid, "new_h1_valid": nr["h1_valid_replicates"], "new_h2_valid": nr["h2_valid_replicates"], "status": parity_status})

write_json(OUT_ROOT / "H1_H2_PARITY_REPORT.json", {
    "status": "PASS",
    "banglaclick_settings_checked": int(sum(r["domain"].startswith("banglaclick:") for r in parity_rows)),
    "baitbuster_settings_repaired": int(sum(r["domain"] == "baitbuster:youtube_human" and r["old_two_class_valid"] < BOOTSTRAP_REPS for r in parity_rows)),
    "rows": parity_rows,
})

stat_lookup = new_lookup
metric_lookup = {(r["domain"], r["model"], int(r["seed"])): r for r in metric_rows}
h1_assess = []
h2_assess = []

for r in h1_screen:
    key = (r["domain"], r["model"], int(r["seed"])); st = stat_lookup[key]
    srcm = metric_lookup[("source_test", r["model"], int(r["seed"]))]; tgtm = metric_lookup[key]
    si, ss = srcm["calibrated"]["calibration_intercept"], srcm["calibrated"]["calibration_slope"]
    ti, ts = tgtm["calibrated"]["calibration_intercept"], tgtm["calibrated"]["calibration_slope"]
    cal_away = False
    if None not in (si, ss, ti, ts):
        cal_away = (abs(ti) > abs(si)) or (abs(ts - 1) > abs(ss - 1))
    if st["h1_inferential_status"] != "ESTIMABLE":
        supported = None
    else:
        nll_evidence = r["nll_relative_change"] >= .10 and st["nll_relative_change_ci95"][0] > 0
        bbs_evidence = r["balanced_brier_relative_change"] >= .10 and st["balanced_brier_relative_change_ci95"][0] > 0
        supported = bool(r["target_auroc"] >= .70 and r["auroc_drop"] <= .05 and (nll_evidence or bbs_evidence) and cal_away)
    h1_assess.append({**r, "inferential_status": st["h1_inferential_status"], "auroc_estimable": st["auroc_estimable"], "nll_relative_change_estimable": st["nll_relative_change_estimable"], "balanced_brier_relative_change_estimable": st["balanced_brier_relative_change_estimable"], "nll_bootstrap_ci95": st["nll_relative_change_ci95"], "balanced_brier_bootstrap_ci95": st["balanced_brier_relative_change_ci95"], "calibration_moved_further_from_ideal": cal_away, "setting_supported": supported})

for r in h2_screen:
    key = (r["domain"], r["model"], int(r["seed"])); st = stat_lookup[key]
    if st["h2_inferential_status"] != "ESTIMABLE":
        supported = None
    else:
        source_positive = r["source_nll_gain"] > 0 and st["h2_source_gain_ci95"][0] > 0
        attenuated = r["target_minus_source_gain"] < 0 and st["h2_target_minus_source_gain_ci95"][1] < 0
        supported = bool(source_positive and attenuated)
    h2_assess.append({**r, "inferential_status": st["h2_inferential_status"], "source_gain_ci95": st["h2_source_gain_ci95"], "target_gain_ci95": st["h2_target_gain_ci95"], "target_minus_source_gain_ci95": st["h2_target_minus_source_gain_ci95"], "setting_supported": supported})


def domain_model_consensus(rows):
    out = []
    rdf = pd.DataFrame(rows)
    for (domain, model), g in rdf.groupby(["domain", "model"], sort=True):
        est = g[g["setting_supported"].notna()]
        supporting = int((est["setting_supported"] == True).sum())
        estimable = int(len(est))
        out.append({
            "domain": domain, "model": model, "supporting_seeds": supporting,
            "estimable_seeds": estimable, "nonestimable_seeds": int(len(g) - estimable), "total_seeds": int(len(g)),
            "consensus_ge_4_of_5": bool(supporting >= 4) if estimable == 5 else None,
        })
    return out

write_json(OUT_ROOT / "H1_RESULTS.json", {
    "schema_version": 2,
    "repair_version": H1H2_REPAIR_VERSION,
    "primary_metrics": "calibrated deployment probabilities",
    "metric_rows": metric_rows,
    "descriptive_support_screen": h1_screen,
    "confirmatory_assessment_rows": h1_assess,
    "domain_model_seed_consensus": domain_model_consensus(h1_assess),
    "estimability_policy": "Endpoint-specific H1 estimability is reported, but overall H1 remains conservative: all three endpoints must be estimable across all 5,000 locked draws; no conditioning on surviving draws.",
    "scientific_non_support_or_nonestimability_is_not_pipeline_failure": True,
})
write_json(OUT_ROOT / "H2_RESULTS.json", {
    "schema_version": 2,
    "repair_version": H1H2_REPAIR_VERSION,
    "definition": "NLL gain = NLL_uncalibrated - NLL_temperature_scaled",
    "rows": h2_screen,
    "confirmatory_assessment_rows": h2_assess,
    "domain_model_seed_consensus": domain_model_consensus(h2_assess),
    "estimability_policy": "H2 NLL gain is evaluated on all 5,000 ordinary cluster-bootstrap draws; single-class draws are not undefined for this proper-loss statistic.",
    "scientific_non_support_or_nonestimability_is_not_pipeline_failure": True,
})
write_json(OUT_ROOT / "SENSITIVITY_RESULTS.json", {
    "schema_version": 1,
    "source_positive_prevalence": source_pi,
    "methods": ["source-prior-standardized class weighting", "source-feature structural composition weighting on token count + code mixing"],
    "metric_rows": metric_rows,
})
print("H1/H2 parity + strict finalization: PASS", flush=True)

H1/H2 parity + strict finalization: PASS



## H3 — validated strict estimability repair

The following code cell is transplanted **unchanged** from the previously validated H3 repair notebook. It is the sole authoritative H3 implementation in the canonical Stage 05.

It preserves all 32 prespecified cue×platform dimensions, requires all 5,000 locked replicates for confirmatory contrast inference, refuses post-hoc dimensionality reduction, and refuses BH-FDR on an incomplete post-hoc family subset.


In [7]:
# ---------- Strict H3 statistical recomputation; no model inference ----------
class ClusterBootstrapPlan:
    def __init__(self, cluster_ids, name):
        clusters = np.asarray(cluster_ids).astype(str)
        if len(clusters) == 0:
            hard_stop("Cannot construct bootstrap plan for empty domain", {"domain":name})
        uniq = np.array(sorted(set(clusters)))
        code = {u:i for i,u in enumerate(uniq)}
        self.row_cluster = np.asarray([code[u] for u in clusters], dtype=np.int32)
        self.cluster_labels = uniq
        self.n_clusters = int(len(uniq))
        self.n_rows = int(len(clusters))
        self.name = str(name)

    def draw_row_counts(self, rng):
        sampled_codes = rng.choice(self.n_clusters, size=self.n_clusters, replace=True)
        multiplicity = np.bincount(sampled_codes, minlength=self.n_clusters)
        return multiplicity[self.row_cluster]

def item_frame(pred):
    return (
        pred[(pred["model"]=="ebm") & (pred["seed"].astype(int)==SEEDS[0])]
        .drop_duplicates("row_id")
        .sort_values("row_id", kind="stable")
        .reset_index(drop=True)
    )

def ebm_seed_frames(pred):
    frames = {}
    canonical = None
    for seed in SEEDS:
        g = (
            pred[(pred["model"]=="ebm") & (pred["seed"].astype(int)==seed)]
            .sort_values("row_id", kind="stable")
            .reset_index(drop=True)
        )
        ids = g["row_id"].astype(str).tolist()
        if canonical is None:
            canonical = ids
        elif ids != canonical:
            hard_stop("EBM seed predictions are not row-aligned", {"seed":seed})
        frames[seed] = g
    return frames

def h3_bundle(item, seed_frames, name):
    ids = item["row_id"].astype(str).tolist()
    y = item["label"].to_numpy(int)
    cues = {cue:(item[cue].to_numpy() > 0) for cue in CUE_FAMILIES}
    losses = []
    for seed in SEEDS:
        g = seed_frames[seed]
        if g["row_id"].astype(str).tolist() != ids:
            hard_stop("H3 EBM seed frame row misalignment", {"domain":name,"seed":seed})
        if not np.array_equal(g["label"].to_numpy(int), y):
            hard_stop("H3 labels differ across EBM seeds", {"domain":name,"seed":seed})
        p = np.clip(g["p_calibrated"].to_numpy(float), 1e-7, 1-1e-7)
        loss = -(y*np.log(p) + (1-y)*np.log(1-p))
        if not np.isfinite(loss).all():
            hard_stop("Non-finite H3 proper loss", {"domain":name,"seed":seed})
        losses.append(loss)
    return {
        "name":name,
        "row_ids":ids,
        "y":y,
        "cues":cues,
        "loss_by_seed":np.vstack(losses),
        "plan":ClusterBootstrapPlan(item["cluster_id"].to_numpy(), name),
        "ones":np.ones(len(item), dtype=np.int16),
    }

def support_record(bundle, cue):
    mask = bundle["cues"][cue]
    clusters = np.asarray(bundle["plan"].cluster_labels)
    row_cluster = bundle["plan"].row_cluster
    present_codes = set(row_cluster[mask].tolist())
    absent_codes = set(row_cluster[~mask].tolist())
    K = bundle["plan"].n_clusters
    k_present = len(present_codes)
    k_absent = len(absent_codes)
    p_no_present = ((K-k_present)/K)**K if K else None
    p_no_absent = ((K-k_absent)/K)**K if K else None
    return {
        "rows":int(bundle["plan"].n_rows),
        "clusters":int(K),
        "cue_present_rows":int(mask.sum()),
        "cue_absent_rows":int((~mask).sum()),
        "clusters_with_cue_present":int(k_present),
        "clusters_with_cue_absent":int(k_absent),
        "analytical_p_resample_missing_cue_present":float(p_no_present),
        "analytical_p_resample_missing_cue_absent":float(p_no_absent),
    }

def group_support_reasons(bundle, cue, row_counts, prefix):
    mask = bundle["cues"][cue]
    reasons = []
    if float(np.asarray(row_counts)[mask].sum()) <= 0:
        reasons.append(f"{prefix}_missing_cue_present")
    if float(np.asarray(row_counts)[~mask].sum()) <= 0:
        reasons.append(f"{prefix}_missing_cue_absent")
    return reasons

def group_gap_1d(values, cue_present, row_counts):
    values = np.asarray(values, float)
    cue_present = np.asarray(cue_present, bool)
    w = np.asarray(row_counts, float)
    wp = w[cue_present]
    wa = w[~cue_present]
    dp = float(wp.sum())
    da = float(wa.sum())
    if dp <= 0 or da <= 0:
        raise ValueError("group_gap_1d called on a non-estimable resample")
    return float(np.dot(values[cue_present], wp)/dp - np.dot(values[~cue_present], wa)/da)

def group_gap_by_seed(loss_by_seed, cue_present, row_counts):
    loss = np.asarray(loss_by_seed, float)
    cue_present = np.asarray(cue_present, bool)
    w = np.asarray(row_counts, float)
    wp = w[cue_present]
    wa = w[~cue_present]
    dp = float(wp.sum())
    da = float(wa.sum())
    if dp <= 0 or da <= 0:
        raise ValueError("group_gap_by_seed called on a non-estimable resample")
    return (loss[:,cue_present] @ wp)/dp - (loss[:,~cue_present] @ wa)/da

def complete_ci(values):
    a = np.asarray(values, float)
    if len(a) != BOOTSTRAP_REPS or not np.isfinite(a).all():
        raise ValueError("Confirmatory CI requires all 5,000 finite locked bootstrap replicates")
    return [float(np.percentile(a, 2.5)), float(np.percentile(a, 97.5))]

def complete_p(values):
    a = np.asarray(values, float)
    if len(a) != BOOTSTRAP_REPS or not np.isfinite(a).all():
        raise ValueError("Confirmatory bootstrap p-value requires all 5,000 finite locked replicates")
    return float(min(1.0, 2*min(np.mean(a <= 0), np.mean(a >= 0))))

def simes_complete(pvals):
    if len(pvals) != 4 or not all(p is not None and np.isfinite(p) for p in pvals):
        return None
    p = np.sort(np.asarray(pvals, float))
    return float(min(1.0, np.min(p*len(p)/np.arange(1, len(p)+1))))

def invalidity_reason_text():
    return (
        "At least one ordinary locked cluster-bootstrap replicate contained zero total "
        "multiplicity in the cue-present or cue-absent group in source and/or target. "
        "The difference-of-group-means statistic is undefined for that replicate. "
        "The locked protocol did not prespecify rejection resampling or conditioning on "
        "valid draws, so confirmatory CI/p are not computed from a post-hoc subset."
    )

src_item = item_frame(SRC_P)
bc_items = {p:item_frame(BC_P[BC_P["platform"].eq(p)]) for p in PLATFORMS}
bb_item = item_frame(BB_P)
src_ebm = ebm_seed_frames(SRC_P)
bc_ebm = {p:ebm_seed_frames(BC_P[BC_P["platform"].eq(p)]) for p in PLATFORMS}
bb_ebm = ebm_seed_frames(BB_P)

src_h3 = h3_bundle(src_item, src_ebm, "source_test")
bc_h3 = {p:h3_bundle(bc_items[p], bc_ebm[p], f"banglaclick:{p}") for p in PLATFORMS}
bb_h3 = h3_bundle(bb_item, bb_ebm, "baitbuster:youtube_human")

# Observed support is written before bootstrap so sparsity is auditable independently.
sparsity_rows = []
for cue in CUE_FAMILIES:
    sparsity_rows.append({"domain":"source_test","cue":cue,**support_record(src_h3,cue)})
    for p in PLATFORMS:
        sparsity_rows.append({"domain":f"banglaclick:{p}","cue":cue,**support_record(bc_h3[p],cue)})
    sparsity_rows.append({"domain":"baitbuster:youtube_human","cue":cue,**support_record(bb_h3,cue)})
pd.DataFrame(sparsity_rows).to_csv(OUT_ROOT/"H3_SPARSITY_REPORT.csv", index=False)

# ---- 32 prespecified BanglaClick cue/platform contrasts ----
rng_h3 = np.random.default_rng(BOOTSTRAP_SEED + 1)
cue_transport = []
print("[H3 repair] BanglaClick cue/platform bootstrap START", flush=True)

for ci, (cue, p) in enumerate([(c,p) for c in CUE_FAMILIES for p in PLATFORMS], 1):
    tgt = bc_h3[p]

    # Point estimate must itself be defined.
    point_reasons = (
        group_support_reasons(src_h3, cue, src_h3["ones"], "source")
        + group_support_reasons(tgt, cue, tgt["ones"], "target")
    )
    if point_reasons:
        point_rd = None
        point_nll = None
        by_seed_json = [None]*len(SEEDS)
    else:
        src_rd = group_gap_1d(src_h3["y"], src_h3["cues"][cue], src_h3["ones"])
        tgt_rd = group_gap_1d(tgt["y"], tgt["cues"][cue], tgt["ones"])
        point_rd = float(tgt_rd - src_rd)
        src_nll_seed = group_gap_by_seed(src_h3["loss_by_seed"], src_h3["cues"][cue], src_h3["ones"])
        tgt_nll_seed = group_gap_by_seed(tgt["loss_by_seed"], tgt["cues"][cue], tgt["ones"])
        by_seed = tgt_nll_seed - src_nll_seed
        point_nll = float(np.mean(by_seed))
        by_seed_json = [float(x) for x in by_seed]

    rd_boot = []
    nll_boot = []
    invalid_reason_counts = defaultdict(int)

    for rep in range(1, BOOTSTRAP_REPS+1):
        sw = src_h3["plan"].draw_row_counts(rng_h3)
        tw = tgt["plan"].draw_row_counts(rng_h3)

        reasons = (
            group_support_reasons(src_h3, cue, sw, "source")
            + group_support_reasons(tgt, cue, tw, "target")
        )
        if reasons:
            invalid_reason_counts["+".join(sorted(reasons))] += 1
        else:
            rd_boot.append(
                group_gap_1d(tgt["y"], tgt["cues"][cue], tw)
                - group_gap_1d(src_h3["y"], src_h3["cues"][cue], sw)
            )
            sd = group_gap_by_seed(src_h3["loss_by_seed"], src_h3["cues"][cue], sw)
            td = group_gap_by_seed(tgt["loss_by_seed"], tgt["cues"][cue], tw)
            nll_boot.append(float(np.mean(td-sd)))

        if rep % 1000 == 0:
            print(f"[H3 repair] contrast {ci}/32 {cue} {p}: {rep}/5000", flush=True)

    invalid = int(sum(invalid_reason_counts.values()))
    valid = BOOTSTRAP_REPS - invalid
    estimable = (not point_reasons) and invalid == 0
    status = "ESTIMABLE" if estimable else "NON_ESTIMABLE_UNDER_LOCKED_BOOTSTRAP"

    if estimable:
        rd_ci = complete_ci(rd_boot)
        rd_p = complete_p(rd_boot)
        nll_ci = complete_ci(nll_boot)
        nll_p = complete_p(nll_boot)
        reason = None
    else:
        rd_ci = rd_p = nll_ci = nll_p = None
        reason = (
            "Observed point statistic is undefined: " + ", ".join(point_reasons)
            if point_reasons else invalidity_reason_text()
        )

    cue_transport.append({
        "cue":cue,
        "target":f"banglaclick:{p}",
        "source_rd":None if point_reasons else float(src_rd),
        "target_rd":None if point_reasons else float(tgt_rd),
        "delta_rd":point_rd,
        "delta_rd_ci95":rd_ci,
        "delta_rd_p":rd_p,
        "ebm_delta_nll_gap_mean_5seeds":point_nll,
        "ebm_delta_nll_gap_ci95":nll_ci,
        "ebm_delta_nll_gap_p":nll_p,
        "ebm_delta_nll_gap_by_seed":by_seed_json,
        "bootstrap_attempts":BOOTSTRAP_REPS,
        "valid_replicates":int(valid),
        "invalid_replicates":int(invalid),
        "invalid_rate":float(invalid/BOOTSTRAP_REPS),
        "invalid_reason_counts":dict(sorted(invalid_reason_counts.items())),
        "inferential_status":status,
        "reason_for_nonestimability":reason,
        "source_support":support_record(src_h3,cue),
        "target_support":support_record(tgt,cue),
    })

# ---- Family Simes + BH-FDR without post-hoc family deletion ----
def family_simes_rows(endpoint_p_key):
    out = {}
    for cue in CUE_FAMILIES:
        rows = [r for r in cue_transport if r["cue"] == cue]
        pvals = [r[endpoint_p_key] for r in rows]
        p = simes_complete(pvals)
        out[cue] = {
            "status":"ESTIMABLE" if p is not None else "NON_ESTIMABLE_INCOMPLETE_FOUR_PLATFORM_FAMILY",
            "simes_p":p,
            "component_p_values":pvals,
            "nonestimable_platforms":[r["target"] for r in rows if r["inferential_status"]!="ESTIMABLE"],
        }
    return out

family_outcome = family_simes_rows("delta_rd_p")
family_reliability = family_simes_rows("ebm_delta_nll_gap_p")

def full_bh(family_rows):
    pvals = [family_rows[c]["simes_p"] for c in CUE_FAMILIES]
    if not all(p is not None and np.isfinite(p) for p in pvals):
        rows = {}
        for cue in CUE_FAMILIES:
            rows[cue] = {
                "simes_p":family_rows[cue]["simes_p"],
                "bh_q":None,
                "reject_q_0_05":None,
                "status":"BH_NOT_RUN_FULL_PRESPECIFIED_FAMILY_INCOMPLETE",
            }
        return {
            "status":"NON_ESTIMABLE_FULL_8_FAMILY_BH",
            "q":FDR_Q,
            "number_prespecified_families":8,
            "number_estimable_family_p_values":int(sum(p is not None for p in pvals)),
            "reason":"BH-FDR was not run on a post-hoc subset; at least one prespecified cue-family Simes p-value is unavailable.",
            "rows":rows,
        }
    reject, qvals, _, _ = multipletests(pvals, alpha=FDR_Q, method="fdr_bh")
    return {
        "status":"ESTIMABLE",
        "q":FDR_Q,
        "number_prespecified_families":8,
        "number_estimable_family_p_values":8,
        "rows":{
            cue:{
                "simes_p":float(p),
                "bh_q":float(q),
                "reject_q_0_05":bool(r),
                "status":"ESTIMABLE",
            }
            for cue,p,q,r in zip(CUE_FAMILIES,pvals,qvals,reject)
        },
    }

family_fdr_outcome_full = full_bh(family_outcome)
family_fdr_reliability_full = full_bh(family_reliability)

# Compatibility dictionaries retain the historical cue-key surface but never
# pretend q-values exist when the full eight-family procedure is non-estimable.
family_fdr_outcome = family_fdr_outcome_full["rows"]
family_fdr_reliability = family_fdr_reliability_full["rows"]

# ---- Full prespecified 32-dimensional BanglaClick outcome Wald test ----
point_vec = np.asarray([
    next(r["delta_rd"] for r in cue_transport if r["cue"]==cue and r["target"]==f"banglaclick:{p}")
    for cue in CUE_FAMILIES for p in PLATFORMS
], dtype=float)

B = np.full((BOOTSTRAP_REPS, 32), np.nan, dtype=float)
global_reason_counts = [defaultdict(int) for _ in range(32)]
rng_global = np.random.default_rng(BOOTSTRAP_SEED + 2)

print("[H3 repair] full 32-D global Wald bootstrap START", flush=True)
for rep in range(BOOTSTRAP_REPS):
    sw = src_h3["plan"].draw_row_counts(rng_global)
    tw = {p:bc_h3[p]["plan"].draw_row_counts(rng_global) for p in PLATFORMS}
    j = 0
    for cue in CUE_FAMILIES:
        source_reasons = group_support_reasons(src_h3, cue, sw, "source")
        for p in PLATFORMS:
            tgt = bc_h3[p]
            reasons = source_reasons + group_support_reasons(tgt, cue, tw[p], "target")
            if reasons:
                global_reason_counts[j]["+".join(sorted(reasons))] += 1
            else:
                B[rep,j] = (
                    group_gap_1d(tgt["y"], tgt["cues"][cue], tw[p])
                    - group_gap_1d(src_h3["y"], src_h3["cues"][cue], sw)
                )
            j += 1
    if (rep+1) % 1000 == 0:
        print(f"[H3 repair] global Wald: {rep+1}/5000", flush=True)

dimension_records = []
j = 0
for cue in CUE_FAMILIES:
    for p in PLATFORMS:
        invalid_count = int((~np.isfinite(B[:,j])).sum())
        dimension_records.append({
            "dimension_index":j,
            "cue":cue,
            "platform":p,
            "invalid_replicates":invalid_count,
            "valid_replicates":int(BOOTSTRAP_REPS-invalid_count),
            "invalid_rate":float(invalid_count/BOOTSTRAP_REPS),
            "invalid_reason_counts":dict(sorted(global_reason_counts[j].items())),
        })
        j += 1

joint_valid = np.isfinite(B).all(axis=1)
point_finite = np.isfinite(point_vec).all()
all_boot_finite = bool(joint_valid.all())

global_result = {
    "status":None,
    "expected_dimension":32,
    "dimension_reduction_performed":False,
    "bootstrap_attempts":BOOTSTRAP_REPS,
    "joint_valid_replicates":int(joint_valid.sum()),
    "joint_invalid_replicates":int((~joint_valid).sum()),
    "dimension_estimability":dimension_records,
    "statistic":None,
    "df":None,
    "p":None,
    "covariance_rank":None,
    "reason":None,
}

if not point_finite:
    global_result["status"] = "NON_ESTIMABLE"
    global_result["reason"] = "At least one prespecified 32-D observed point component is undefined."
elif not all_boot_finite:
    global_result["status"] = "NON_ESTIMABLE"
    global_result["reason"] = (
        "At least one of the 5,000 locked joint cluster-bootstrap vectors contains an undefined "
        "prespecified dimension. No dimensions were dropped and no reduced post-hoc Wald test was run."
    )
else:
    cov = np.atleast_2d(np.cov(B, rowvar=False))
    rank = int(np.linalg.matrix_rank(cov))
    global_result["covariance_rank"] = rank
    if rank != 32:
        global_result["status"] = "NON_ESTIMABLE"
        global_result["reason"] = (
            f"The complete finite 32×32 bootstrap covariance has rank {rank}, not 32. "
            "No rank-reduced post-hoc Wald test was substituted."
        )
    else:
        try:
            wald = float(point_vec @ np.linalg.solve(cov, point_vec))
        except np.linalg.LinAlgError:
            global_result["status"] = "NON_ESTIMABLE"
            global_result["reason"] = "The full 32×32 covariance could not be stably solved."
        else:
            global_result.update({
                "status":"ESTIMABLE",
                "statistic":wald,
                "df":32,
                "p":float(chi2.sf(wald, 32)),
                "reason":None,
            })

# ---- Independent BaitBuster replication; same strict estimability reporting ----
bb_rep = []
rng_rep = np.random.default_rng(BOOTSTRAP_SEED + 3)
print("[H3 repair] BaitBuster replication bootstrap START", flush=True)

for cue_i, cue in enumerate(CUE_FAMILIES, 1):
    point_reasons = (
        group_support_reasons(src_h3, cue, src_h3["ones"], "source")
        + group_support_reasons(bb_h3, cue, bb_h3["ones"], "target")
    )
    if point_reasons:
        point_rd = point_nll = None
        by_seed_json = [None]*len(SEEDS)
    else:
        src_rd = group_gap_1d(src_h3["y"], src_h3["cues"][cue], src_h3["ones"])
        tgt_rd = group_gap_1d(bb_h3["y"], bb_h3["cues"][cue], bb_h3["ones"])
        point_rd = float(tgt_rd-src_rd)
        src_nll = group_gap_by_seed(src_h3["loss_by_seed"], src_h3["cues"][cue], src_h3["ones"])
        tgt_nll = group_gap_by_seed(bb_h3["loss_by_seed"], bb_h3["cues"][cue], bb_h3["ones"])
        by_seed = tgt_nll-src_nll
        point_nll = float(np.mean(by_seed))
        by_seed_json = [float(x) for x in by_seed]

    rd_boot = []
    nll_boot = []
    reason_counts = defaultdict(int)

    for rep in range(1, BOOTSTRAP_REPS+1):
        sw = src_h3["plan"].draw_row_counts(rng_rep)
        tw = bb_h3["plan"].draw_row_counts(rng_rep)
        reasons = (
            group_support_reasons(src_h3, cue, sw, "source")
            + group_support_reasons(bb_h3, cue, tw, "target")
        )
        if reasons:
            reason_counts["+".join(sorted(reasons))] += 1
        else:
            rd_boot.append(
                group_gap_1d(bb_h3["y"], bb_h3["cues"][cue], tw)
                - group_gap_1d(src_h3["y"], src_h3["cues"][cue], sw)
            )
            sd = group_gap_by_seed(src_h3["loss_by_seed"], src_h3["cues"][cue], sw)
            td = group_gap_by_seed(bb_h3["loss_by_seed"], bb_h3["cues"][cue], tw)
            nll_boot.append(float(np.mean(td-sd)))
        if rep % 1000 == 0:
            print(f"[H3 repair] BaitBuster {cue_i}/8 {cue}: {rep}/5000", flush=True)

    invalid = int(sum(reason_counts.values()))
    valid = BOOTSTRAP_REPS-invalid
    estimable = (not point_reasons) and invalid == 0
    bb_rep.append({
        "cue":cue,
        "delta_rd":point_rd,
        "delta_rd_ci95":complete_ci(rd_boot) if estimable else None,
        "delta_rd_p":complete_p(rd_boot) if estimable else None,
        "ebm_delta_nll_gap_mean_5seeds":point_nll,
        "ebm_delta_nll_gap_ci95":complete_ci(nll_boot) if estimable else None,
        "ebm_delta_nll_gap_p":complete_p(nll_boot) if estimable else None,
        "ebm_delta_nll_gap_by_seed":by_seed_json,
        "bootstrap_attempts":BOOTSTRAP_REPS,
        "valid_replicates":int(valid),
        "invalid_replicates":int(invalid),
        "invalid_rate":float(invalid/BOOTSTRAP_REPS),
        "invalid_reason_counts":dict(sorted(reason_counts.items())),
        "inferential_status":"ESTIMABLE" if estimable else "NON_ESTIMABLE_UNDER_LOCKED_BOOTSTRAP",
        "reason_for_nonestimability":None if estimable else (
            "Observed point statistic is undefined: " + ", ".join(point_reasons)
            if point_reasons else invalidity_reason_text()
        ),
        "source_support":support_record(src_h3,cue),
        "target_support":support_record(bb_h3,cue),
    })

nonestimable_bc = [r for r in cue_transport if r["inferential_status"]!="ESTIMABLE"]
nonestimable_bb = [r for r in bb_rep if r["inferential_status"]!="ESTIMABLE"]

H3_CORRECTED = {
    "schema_version":3,
    "repair_version":REPAIR_VERSION,
    "repair_scope":"H3 statistical layer only; immutable Stage-05 predictions consumed read-only",
    "source_stage05_manifest_sha256":source_manifest_sha,
    "prediction_artifact_sha256":SOURCE_IDENTITY,
    "bootstrap_replicates":BOOTSTRAP_REPS,
    "bootstrap_seed":BOOTSTRAP_SEED,
    "ci":"95% percentile; confirmatory only when all 5,000 locked replicates are finite",
    "cluster_rule":"unchanged from Stage 05: strongest available source entity; source TEST=row; BanglaClick=stored SOURCE_URL-derived cluster_id; BaitBuster=stored channel_id-derived cluster_id",
    "primary":"prespecified cue–outcome and cue-conditioned proper-loss transport",
    "proper_loss":"NLL on frozen temperature-scaled deployment probabilities",
    "estimability_policy":{
        "conditional_valid_only_bootstrap_used":False,
        "rejection_resampling_used":False,
        "posthoc_dimension_reduction_used":False,
        "contrast_rule":"Any invalid locked replicate => confirmatory CI/p NON_ESTIMABLE.",
        "global_rule":"All 32 dimensions and all 5,000 joint replicates plus full-rank 32×32 covariance are required.",
        "bh_rule":"BH-FDR is not run on a post-hoc subset of the eight prespecified cue families.",
    },
    "global_banglaclick_outcome_wald":global_result,
    "global_banglaclick_wald":global_result,
    "banglaclick_cue_platform_contrasts":cue_transport,
    "family_simes_outcome":family_outcome,
    "family_simes_reliability":family_reliability,
    "family_fdr_outcome_full":family_fdr_outcome_full,
    "family_fdr_reliability_full":family_fdr_reliability_full,
    "family_fdr_outcome":family_fdr_outcome,
    "family_fdr_reliability":family_fdr_reliability,
    "family_fdr":family_fdr_outcome,
    "baitbuster_replication":bb_rep,
    "nonestimable_banglaclick_contrast_count":len(nonestimable_bc),
    "nonestimable_baitbuster_replication_count":len(nonestimable_bb),
    "causal_language":False,
    "interpretation":"association/transport of predictive outcome and proper-loss relationships under observed distribution shift",
}
write_json(OUT_ROOT/"H3_RESULTS.json", H3_CORRECTED)
pd.DataFrame(cue_transport).to_csv(OUT_ROOT/"H3_CUE_PLATFORM_CONTRASTS_CORRECTED.csv", index=False)
pd.DataFrame(bb_rep).to_csv(OUT_ROOT/"H3_BAITBUSTER_REPLICATION_CORRECTED.csv", index=False)

print(
    "[H3 repair] COMPLETE — "
    f"BanglaClick non-estimable contrasts={len(nonestimable_bc)}/32; "
    f"global={global_result['status']}; "
    f"BaitBuster non-estimable cues={len(nonestimable_bb)}/8",
    flush=True,
)


[H3 repair] BanglaClick cue/platform bootstrap START


[H3 repair] contrast 1/32 curiosity_information_withholding news: 1000/5000


[H3 repair] contrast 1/32 curiosity_information_withholding news: 2000/5000


[H3 repair] contrast 1/32 curiosity_information_withholding news: 3000/5000


[H3 repair] contrast 1/32 curiosity_information_withholding news: 4000/5000


...

[H3 repair] BaitBuster 8/8 vague_deictic_reference: 2000/5000


[H3 repair] BaitBuster 8/8 vague_deictic_reference: 3000/5000


[H3 repair] BaitBuster 8/8 vague_deictic_reference: 4000/5000


[H3 repair] BaitBuster 8/8 vague_deictic_reference: 5000/5000


[H3 repair] COMPLETE — BanglaClick non-estimable contrasts=12/32; global=NON_ESTIMABLE; BaitBuster non-estimable cues=3/8



## H3-P — strict finalization from immutable pair-level inference cache

No model is loaded. Pair construction and `delta_z` model inference are reused byte-for-byte from the successful Stage-05 run. Before bootstrap finalization, point estimates are required to match the original H3-P result table. The final bootstrap uses multiplicity weights and all 5,000 locked draws.


In [8]:
OLD_H3P = json.loads((SOURCE_ROOT / "H3_PERTURBATION_RESULTS.json").read_text(encoding="utf-8"))
old_h3p_lookup = {(r["cue"], r["model"], int(r["seed"]), r["target"]): r for r in OLD_H3P.get("rows", [])}

source_groups = {
    (cue, model, int(seed)): sg.sort_values("row_id", kind="stable").reset_index(drop=True)
    for (cue, model, seed), sg in PERT[PERT.domain == "source_test"].groupby(["cue", "model", "seed"], sort=True)
}
target_group_records = []
for (cue, model, seed), sg in PERT[PERT.domain == "source_test"].groupby(["cue", "model", "seed"], sort=True):
    for dname, tg in PERT[(PERT.domain != "source_test") & (PERT.cue == cue) & (PERT.model == model) & (PERT.seed == seed)].groupby("domain", sort=True):
        target_group_records.append((cue, model, int(seed), dname, tg.sort_values("row_id", kind="stable").reset_index(drop=True)))

if len(target_group_records) != len(PERT_FAMS) * 2 * len(SEEDS) * 5:
    hard_stop("Unexpected H3-P contrast count", {"observed": len(target_group_records), "expected": len(PERT_FAMS) * 2 * len(SEEDS) * 5})

# Point-estimate parity is checked before any bootstrap work.
point_parity_rows = []
for cue, model, seed, dname, tg in target_group_records:
    sg = source_groups[(cue, model, seed)]
    ps = float(np.mean(sg.delta_z.to_numpy(float))); pt = float(np.mean(tg.delta_z.to_numpy(float)))
    key = (cue, model, seed, dname)
    is_branch_news = bool(BRANCH_RULE) and dname == "banglaclick:news"
    if key not in old_h3p_lookup:
        hard_stop("Original H3-P parity reference missing contrast", {"contrast": key})
    old = old_h3p_lookup[key]
    ok = all(np.isclose(a, b, rtol=0.0, atol=1e-12) for a, b in [
        (ps, old["source_mean_delta_z"]),
        (pt, old["target_mean_delta_z"]),
        (pt - ps, old["target_minus_source_delta_z"]),
    ])
    if (not is_branch_news) and (not ok):
        hard_stop("H3-P point-estimate parity failed", {"contrast": key})
    point_parity_rows.append({"cue": cue, "model": model, "seed": seed, "target": dname, "point_parity": (None if is_branch_news else True), "status": ("BRANCH_EXPECTED_DIFFERENCE" if is_branch_news else "EXACT_NUMERICAL_PARITY")})

write_json(OUT_ROOT / "H3P_POINT_PARITY.json", {"status": "PASS", "contrasts": len(point_parity_rows), "rows": point_parity_rows})
print(f"H3-P point parity: PASS ({len(point_parity_rows)}/{len(point_parity_rows)})", flush=True)

plan_cache = {}
def _pert_plan(domain, cue, frame):
    key = (domain, cue, tuple(frame.row_id.astype(str).tolist()))
    if key not in plan_cache:
        plan_cache[key] = ClusterBootstrapPlan(frame.cluster_id.to_numpy(), f"{domain}|{cue}")
    return plan_cache[key]

rngp = np.random.default_rng(BOOTSTRAP_SEED + 4)
pert_results = []
pert_start = time.monotonic()
for ci, (cue, model, seed, dname, tg) in enumerate(target_group_records, 1):
    sg = source_groups[(cue, model, seed)]
    spl = _pert_plan("source_test", cue, sg); tpl = _pert_plan(dname, cue, tg)
    sv = sg.delta_z.to_numpy(float); tv = tg.delta_z.to_numpy(float)
    point_s = float(np.mean(sv)); point_t = float(np.mean(tv))
    diffs = []
    invalid = 0
    for rep in range(1, BOOTSTRAP_REPS + 1):
        sw = spl.draw_row_counts(rngp); tw = tpl.draw_row_counts(rngp)
        d = _weighted_mean(tv, tw) - _weighted_mean(sv, sw)
        if np.isfinite(d):
            diffs.append(float(d))
        else:
            invalid += 1
        if rep % 100 == 0:
            _phase_guard("H3-P bootstrap", pert_start, rep, BOOTSTRAP_REPS, {"contrast": ci})
        if rep % BOOTSTRAP_PROGRESS_EVERY == 0:
            _progress("H3-P bootstrap", rep, BOOTSTRAP_REPS, pert_start, f"contrast={ci}/{len(target_group_records)}")
    estimable = invalid == 0 and len(diffs) == BOOTSTRAP_REPS
    pert_results.append({
        "cue": cue, "model": model, "seed": int(seed), "target": dname,
        "source_mean_delta_z": point_s,
        "target_mean_delta_z": point_t,
        "target_minus_source_delta_z": point_t - point_s,
        "bootstrap_attempts": BOOTSTRAP_REPS,
        "valid_replicates": int(len(diffs)),
        "invalid_replicates": int(invalid),
        "inferential_status": "ESTIMABLE" if estimable else "NON_ESTIMABLE_UNDER_LOCKED_BOOTSTRAP",
        "ci95": complete_ci(diffs) if estimable else None,
        "p": complete_p(diffs) if estimable else None,
    })

# Because delta_z is finite and every ordinary cluster draw has positive total multiplicity,
# the repaired implementation should have exact parity with the original H3-P statistics.
h3p_parity = []
for r in pert_results:
    key = (r["cue"], r["model"], int(r["seed"]), r["target"])
    old = old_h3p_lookup[key]
    is_branch_news = bool(BRANCH_RULE) and r["target"] == "banglaclick:news"
    ci_ok = r["ci95"] is not None and len(old.get("ci95", [])) == 2 and all(np.isclose(a, b, rtol=0.0, atol=1e-12) for a, b in zip(r["ci95"], old["ci95"]))
    p_ok = r["p"] is not None and np.isclose(r["p"], old["p"], rtol=0.0, atol=1e-12)
    if (not is_branch_run) and not (ci_ok and p_ok and r["valid_replicates"] == BOOTSTRAP_REPS):
        hard_stop("H3-P full statistical parity failed", {"contrast": key, "ci_ok": ci_ok, "p_ok": p_ok, "valid": r["valid_replicates"]})
    h3p_parity.append({"cue": r["cue"], "model": r["model"], "seed": r["seed"], "target": r["target"], "status": ("BRANCH_EXPECTED_DIFFERENCE" if is_branch_run else "EXACT_NUMERICAL_PARITY")})

write_json(OUT_ROOT / "H3P_STATISTICAL_PARITY.json", {"status": "PASS", "contrasts": len(h3p_parity), "rows": h3p_parity})
write_json(OUT_ROOT / "H3_PERTURBATION_RESULTS.json", {
    "schema_version": 2,
    "finalization_version": H3P_FINALIZATION_VERSION,
    "response": "delta_z = normalized-original logit - cue-neutralized logit",
    "causal_real_world_claim": False,
    "pair_representation": "Immutable pair-level inference cache from successful Stage 05; original and neutralized inputs share P0 normalization.",
    "models": ["ebm", "banglabert"],
    "families": PERT_FAMS,
    "estimability_policy": "All 5,000 locked ordinary cluster-bootstrap draws are required; no NaN filtering or rejection resampling.",
    "rows": pert_results,
    "pairs_sha256": sha256_file(OUT_ROOT / "H3_PERTURBATION_PAIRS.parquet"),
    "exclusions_sha256": sha256_file(OUT_ROOT / "H3_PERTURBATION_EXCLUSIONS.csv"),
})
print(f"H3-P strict finalization: PASS contrasts={len(pert_results)} elapsed={time.monotonic()-pert_start:.1f}s", flush=True)

H3-P point parity: PASS (150/150)


[H3-P bootstrap] 1000/5000 elapsed=0.0s eta=0.1s contrast=1/150


[H3-P bootstrap] 2000/5000 elapsed=0.1s eta=0.1s contrast=1/150


[H3-P bootstrap] 3000/5000 elapsed=0.1s eta=0.1s contrast=1/150


[H3-P bootstrap] 4000/5000 elapsed=0.1s eta=0.0s contrast=1/150


...

[H3-P bootstrap] 2000/5000 elapsed=21.2s eta=31.8s contrast=150/150


[H3-P bootstrap] 3000/5000 elapsed=21.2s eta=14.1s contrast=150/150


[H3-P bootstrap] 4000/5000 elapsed=21.2s eta=5.3s contrast=150/150


[H3-P bootstrap] 5000/5000 elapsed=21.3s eta=0.0s contrast=150/150


H3-P strict finalization: PASS contrasts=150 elapsed=21.3s


## Final statistical summary, provenance seal, and PASS

In [9]:
h3_now = json.loads((OUT_ROOT / "H3_RESULTS.json").read_text(encoding="utf-8"))
h1_now = json.loads((OUT_ROOT / "H1_RESULTS.json").read_text(encoding="utf-8"))
h2_now = json.loads((OUT_ROOT / "H2_RESULTS.json").read_text(encoding="utf-8"))
h3p_now = json.loads((OUT_ROOT / "H3_PERTURBATION_RESULTS.json").read_text(encoding="utf-8"))

h1_nonest = sum(r["h1_inferential_status"] != "ESTIMABLE" for r in stat_rows)
h1_auroc_nonest = sum(not r["auroc_estimable"] for r in stat_rows)
h1_nll_nonest = sum(not r["nll_relative_change_estimable"] for r in stat_rows)
h1_bb_nonest = sum(not r["balanced_brier_relative_change_estimable"] for r in stat_rows)
h2_nonest = sum(r["h2_inferential_status"] != "ESTIMABLE" for r in stat_rows)
h3_nonest = int(h3_now["nonestimable_banglaclick_contrast_count"])
h3p_nonest = sum(r["inferential_status"] != "ESTIMABLE" for r in h3p_now["rows"])

STATISTICAL_SUMMARY = {
    "schema_version": 3,
    "canonical_version": CANONICAL_VERSION,
    "bootstrap_replicates": BOOTSTRAP_REPS,
    "bootstrap_seed": BOOTSTRAP_SEED,
    "ci": "95% percentile; confirmatory only when all locked draws required by the statistic are finite",
    "cluster_rule": "unchanged: strongest available source entity; source TEST=row; BanglaClick=SOURCE_URL-derived cluster_id; BaitBuster=channel_id-derived cluster_id",
    "h1_h2_repair_version": H1H2_REPAIR_VERSION,
    "h1_h2_bootstrap": stat_rows,
    "h1_overall_estimability": {
        "rule": "ESTIMABLE only when AUROC drop, relative NLL change, and relative balanced-Brier change are each estimable on all 5,000 locked draws.",
        "estimable_settings": int(len(stat_rows) - h1_nonest),
        "nonestimable_settings": int(h1_nonest),
    },
    "h1_endpoint_estimability": {
        "auroc": {"estimable_settings": int(len(stat_rows) - h1_auroc_nonest), "nonestimable_settings": int(h1_auroc_nonest)},
        "nll_relative_change": {"estimable_settings": int(len(stat_rows) - h1_nll_nonest), "nonestimable_settings": int(h1_nll_nonest)},
        "balanced_brier_relative_change": {"estimable_settings": int(len(stat_rows) - h1_bb_nonest), "nonestimable_settings": int(h1_bb_nonest)},
    },
    "h1_nonestimable_settings": int(h1_nonest),
    "h2_nonestimable_settings": int(h2_nonest),
    "h3_repair_version": H3_REPAIR_VERSION,
    "h3_global": h3_now["global_banglaclick_outcome_wald"],
    "h3_nonestimable_banglaclick_contrasts": h3_nonest,
    "h3_family_fdr_outcome_full": h3_now["family_fdr_outcome_full"],
    "h3_family_fdr_reliability_full": h3_now["family_fdr_reliability_full"],
    "h3p_finalization_version": H3P_FINALIZATION_VERSION,
    "h3p_nonestimable_contrasts": int(h3p_nonest),
    "fdr_method": "Separate BH control over all 8 prespecified cue-family Simes p-values; never run on a post-hoc subset.",
    "fdr_q": FDR_Q,
    "development_statistical_outputs_inherited": False,
    "scientific_nonestimability_is_not_pipeline_failure": True,
}
write_json(OUT_ROOT / "STATISTICAL_SUMMARY.json", STATISTICAL_SUMMARY)

RUN_METADATA = {
    "schema_version": 2,
    "stage_id": STAGE_ID,
    "canonical_version": CANONICAL_VERSION,
    "execution_utc": utc_now(),
    "python": sys.version,
    "platform": platform.platform(),
    "runtime": {"cpu_count": os.cpu_count(), "gpu_required": False},
    "packages": {n: pkg(n) for n in ["numpy", "pandas", "pyarrow", "scipy", "statsmodels", "scikit-learn"]},
    "seeds": SEEDS,
    "bootstrap_reps": BOOTSTRAP_REPS,
    "git": git_state(),
    "cache_source_manifest_sha256": SOURCE_MANIFEST_SHA256,
    "cache_identity": CACHE_IDENTITY,
    "model_inference_rerun": False,
    "calibration_refit": False,
    "target_tuning": False,
}
write_json(OUT_ROOT / "RUN_METADATA.json", RUN_METADATA)

summary_lines = [
    "# Stage 05 summary",
    "",
    "- Immutable prediction and H3-P inference cache: verified and reused; no model inference rerun.",
    f"- Branch mode: {bool(BRANCH_RULE)}; rule={BRANCH_RULE}.",
    f"- H1 overall non-estimable settings: {h1_nonest}/{len(stat_rows)}.",
    f"- H1 endpoint non-estimable settings — AUROC: {h1_auroc_nonest}/{len(stat_rows)}; relative NLL: {h1_nll_nonest}/{len(stat_rows)}; relative balanced Brier: {h1_bb_nonest}/{len(stat_rows)}.",
    f"- H2 non-estimable settings: {h2_nonest}/{len(stat_rows)}.",
    f"- H3 non-estimable BanglaClick cue×platform contrasts: {h3_nonest}/32.",
    f"- H3 full 32-D global Wald status: {h3_now['global_banglaclick_outcome_wald']['status']}.",
    f"- H3-P non-estimable contrasts: {h3p_nonest}/{len(h3p_now['rows'])}.",
    "- No post-hoc H3 dimensionality reduction or incomplete-family BH-FDR was performed.",
    "- Scientific non-support/non-estimability does not cause computational stage failure.",
]
write_text(OUT_ROOT / "STAGE05_SUMMARY.md", "\n".join(summary_lines) + "\n")

required = [
    "ENVIRONMENT_PREFLIGHT.json", "INFERENCE_CACHE_PROVENANCE.json", "CACHE_SCHEMA_VALIDATION.json",
    "LOCKED_SOURCE_TEST_PREDICTIONS.parquet", "BANGLACLICK_PLATFORM_PREDICTIONS.parquet", "BAITBUSTER_HUMAN_PREDICTIONS.parquet",
    "H1_H2_PARITY_REPORT.json", "H1_RESULTS.json", "H2_RESULTS.json", "SENSITIVITY_RESULTS.json",
    "H3_RESULTS.json", "H3_SPARSITY_REPORT.csv", "H3_CUE_PLATFORM_CONTRASTS_CORRECTED.csv", "H3_BAITBUSTER_REPLICATION_CORRECTED.csv",
    "H3_PERTURBATION_PAIRS.parquet", "H3_PERTURBATION_EXCLUSIONS.csv", "H3P_POINT_PARITY.json", "H3P_STATISTICAL_PARITY.json", "H3_PERTURBATION_RESULTS.json",
    "STATISTICAL_SUMMARY.json", "RUN_METADATA.json", "STAGE05_SUMMARY.md",
]
missing = [name for name in required if not (OUT_ROOT / name).is_file()]
if missing:
    hard_stop("Canonical Stage-05 artifact contract incomplete", {"missing": missing})

manifest = {
    "schema_version": 2,
    "stage_id": STAGE_ID,
    "canonical_stage05": (False if bool(BRANCH_RULE) else True),
    "branch_mode": bool(BRANCH_RULE),
    "branch_rule": BRANCH_RULE,
    "base_canonical_stage05_manifest_sha256": os.environ.get("SWARABYANJAN_BASE_STAGE05_MANIFEST_SHA256"),
    "canonical_version": CANONICAL_VERSION,
    "protocol_version": PROTOCOL_VERSION,
    "source_inference_cache_manifest_sha256": SOURCE_MANIFEST_SHA256,
    "upstream_stage03_checksum_manifest_sha256": SOURCE_MANIFEST.get("upstream_stage03_checksum_manifest_sha256"),
    "upstream_stage04_checksum_manifest_sha256": SOURCE_MANIFEST.get("upstream_stage04_checksum_manifest_sha256"),
    "frozen_content_identity_sha256": SOURCE_MANIFEST.get("frozen_content_identity_sha256"),
    "immutable_cache_identity": CACHE_IDENTITY,
    "scientific_boundary": {
        "model_inference_reused": True,
        "model_inference_rerun": False,
        "calibration_refit": False,
        "h1_h2_statistics_recomputed": True,
        "h3_validated_repair_recomputed": True,
        "h3p_statistics_recomputed_from_immutable_pair_inference": True,
        "old_development_statistics_inherited": False,
    },
    "artifacts": {name: sha256_file(OUT_ROOT / name) for name in required},
}
manifest["manifest_payload_sha256"] = hashlib.sha256(cjb(manifest)).hexdigest()
write_json(OUT_ROOT / "STAGE05_ARTIFACT_MANIFEST.json", manifest)

entries = []
for p in sorted(OUT_ROOT.iterdir(), key=lambda x: x.name):
    if p.is_file() and p.name not in {"STAGE05_SHA256SUMS.txt", "STAGE_STATUS.json"}:
        entries.append((p.name, sha256_file(p)))
write_text(OUT_ROOT / "STAGE05_SHA256SUMS.txt", "\n".join(f"{h}  {name}" for name, h in entries) + "\n")

set_status(
    "PASS",
    "Stage-05 branch external validation/H1/H2/H3/H3-P finalization completed" if bool(BRANCH_RULE) else "Canonical Stage-05 external validation/H1/H2/H3/H3-P finalization completed",
    {
        "model_inference_rerun": False,
        "h1_nonestimable_settings": int(h1_nonest),
        "h1_endpoint_nonestimable_settings": {"auroc": int(h1_auroc_nonest), "nll_relative_change": int(h1_nll_nonest), "balanced_brier_relative_change": int(h1_bb_nonest)},
        "h2_nonestimable_settings": int(h2_nonest),
        "h3_nonestimable_banglaclick_contrasts": int(h3_nonest),
        "h3_global_status": h3_now["global_banglaclick_outcome_wald"]["status"],
        "h3p_nonestimable_contrasts": int(h3p_nonest),
        "output_root": str(OUT_ROOT),
    },
)
print("STAGE 05 CANONICAL FINAL — PASS", flush=True)

STAGE 05 STATUS: PASS — Stage-05 branch external validation/H1/H2/H3/H3-P finalization completed


STAGE 05 CANONICAL FINAL — PASS
